# Main Data 

## Imports, Helpers, Parameters

### Imports

In [638]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import wrap
import numpy as np
import re
import os
from pathlib import Path
from collections import Counter


### Helper Functions

In [639]:
#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text


### Adjust Parameters

In [640]:
# Frequency mapping. Assuming a 52 week year with 5 working days per week, these are corresponding survey questions::
# 1 Once per year or less (Assuming 1 time per year)
# 2 More than once per year (Assuming 3 times per year)
# 3 More than once per month (Assuming 48 times per year, 3 times per month)
# 4 More than once per week (Assuming 130 times per year, 2.5 times per week)
# 5 Daily
# 6 Several times per day (Assuming 3 times per day)
# 7 Hourly or more often (Assuming 12 times per day, 1.5 times per hour)
frequency_weights = {
    1: 1 / 260,
    2: 3 / 260,
    3: 48 / 260,
    4: 130 / 260,
    5: 1,
    6: 3,
    7: 12
}

# Adjust inflation for scraped wage data (Jan 2020 to now)
jan_2020_inflation_factor = 1.24

# Adjust inflation for BLS wage data (May 2015 to now)
may_2015_inflation_factor = 1.36


## CSV Inputs and Config

In [642]:
df = pd.read_excel("../Job Zones.xlsx")
df.to_csv("../data/job_zones_v30.1.csv", index=False)

In [570]:
# Change to true to get each data frame saved to a csv after every step to a folder named merged_data_files
save_files_each_step = False
run_name = "first_pass_eco_tasks_2025.csv"

In [571]:
data_dir = Path("../data")

runs = {
    "first_pass_microsoft.csv": "iwa_metrics.csv",
    "first_pass_aei_v1.csv": "task_pct_v1.csv",
    "first_pass_aei_v2.csv": "task_pct_v2.csv",
    "first_pass_mcp_v1.csv": "task_results_2025-04-24.csv",
    "first_pass_mcp_v2.csv": "task_results_2025-05-24.csv",
    "first_pass_mcp_v3.csv": "task_results_2025-07-23.csv",
    "first_pass_aei_v3.csv": "task_pct_v3.csv",
    "first_pass_aei_api_v3.csv": "task_pct_api_v3.csv",
    "first_pass_aei_v4.csv": "task_pct_v4.csv",
    "first_pass_aei_api_v4.csv": "task_pct_api_v4.csv",
    "first_pass_aei_v5.csv": "task_pct_v5.csv",
    "first_pass_aei_api_v5.csv": "task_pct_api_v5.csv",
    "first_pass_mcp_v4.csv": "task_results_2026-02-18.csv",
    "first_pass_eco_tasks_2015.csv": "task_pct_eco_2015.csv",
    "first_pass_eco_tasks_2025.csv": "task_pct_eco_2025.csv"
}

runs_v3_and_up = {
    "first_pass_aei_v3.csv": "aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv",
    "first_pass_aei_api_v3.csv": "aei_raw_1p_api_2025-08-04_to_2025-08-11.csv",
    "first_pass_aei_v4.csv": "aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv",
    "first_pass_aei_api_v4.csv": "aei_raw_1p_api_2025-11-13_to_2025-11-20.csv",
    "first_pass_aei_v5.csv": "aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv",
    "first_pass_aei_api_v5.csv": "aei_raw_1p_api_2026-02-05_to_2026-02-12.csv",
}

# ---- AUTO DETECT TYPE ----

aei_run = "aei" in run_name.lower()
mcp_run = "mcp" in run_name.lower()
microsoft_run = "microsoft" in run_name.lower()
eco_run_2015 = "first_pass_eco_tasks_2015.csv" == run_name
eco_run_2025 = "first_pass_eco_tasks_2025.csv" == run_name

aei_v3_and_up = aei_run and any(v in run_name for v in ["v3", "v4", "v5"])

# ---- SET INPUT FILE ----

input_file = data_dir / runs[run_name]

# ---- SET SPECIAL AEI RAW FILE (if needed) ----

if aei_v3_and_up:
    aei_v3_and_up_pct_df = data_dir / runs_v3_and_up[run_name]
else:
    aei_v3_and_up_pct_df = None

# ---- SET NOTEBOOK STEP INPUTS ----

pct_df = input_file if aei_run else None
mcp_data = input_file if mcp_run else None
microsoft_data = input_file if microsoft_run else None
eco_data_2015 = input_file if eco_run_2015 else None
eco_data_2025 = input_file if eco_run_2025 else None
# ---- OUTPUT FILE ----

output_file_main_1 = data_dir / run_name

print("Run type:")
print("AEI:", aei_run)
print("MCP:", mcp_run)
print("Microsoft:", microsoft_run)
print("AEI v3+:", aei_v3_and_up)
print("Input file:", input_file)
print("Output file:", output_file_main_1)

Run type:
AEI: False
MCP: False
Microsoft: False
AEI v3+: False
Input file: ..\data\task_pct_eco_2025.csv
Output file: ..\data\first_pass_eco_tasks_2025.csv


## Step 1: Map Anthropic Task %s to O*NET v20.1 Task Statements

In [572]:
if mcp_run or microsoft_run:
    save_files_each_step = False

if save_files_each_step:
    import os
    os.makedirs("../merged_data_files", exist_ok=True)

#### AEI v3+ Pre Processing

In [573]:
if aei_v3_and_up:
    df = pd.read_csv(aei_v3_and_up_pct_df)

    filtered = df[
        (df["geo_id"] == "GLOBAL") &
        (df["facet"] == "onet_task") &
        (df["variable"] == "onet_task_pct")
    ][["cluster_name", "value"]].rename(columns={"cluster_name": "task_name", "value": "pct"})

    filtered.to_csv(pct_df, index=False)

#### ONET Version

In [574]:
def pct_to_onet_tasks(pct_df, task_statements_df) -> pd.DataFrame:
    """
    Description:
        This loads in the tasks and percentage of occurrences from the Anthropic data, and merges it with the tasks statement data. 
        It normalizes the percents based on a weighted and non weighted approach.
        See documentation for more details.

    Args:
        pct_df (pd.DataFrame): DataFrame containing the Anthropic data of percent occurances of every task in their conversation data
        task_statements_df (pd.DataFrame): DataFrame containing O*NET tasks and SOC titles.
    
    Returns:
        pd.DataFrame: Updated DataFrame with percentage of occurrences added.
    """

    task_statements_df.rename(columns={
    "O*NET-SOC Code": "soc_code_2010",
    "Title": "title",
    "Task ID": "task_id",
    "Task": "task",
    "Task Type": "task_type",
    "Incumbents Responding": "n_responding",
    "Date": "date",
    "Domain Source": "domain_source",
    }, inplace=True)

    # Normalize task columns
    pct_df["task_normalized_temp"] = pct_df["task_name"].apply(normalize_text)
    task_statements_df["task_normalized"] = task_statements_df["task"].apply(normalize_text)
    # task_statements_df["task_normalized"] = task_statements_df["task"].str.lower().str.strip()

    pct_df = pct_df.groupby("task_normalized_temp", as_index=False).agg({
    "task_name": "first",  # Keep the first task name
    "pct": "sum"  # Sum the percentages for duplicates
    })
    
    # Merge dfs
    merged = pct_df.merge(
        task_statements_df,
        left_on="task_normalized_temp",
        right_on="task_normalized",
        how="left"
    )
    
    # Calculate weighted and normalized percentages
    merged["n_occurrences"] = merged.groupby("task_normalized")["title"].transform("nunique")
    merged["pct_weighted"] = 100 * merged["pct"] / merged["pct"].sum()
    merged["pct_normalized"] = 100 * (merged["pct"] / merged["n_occurrences"]) / (merged["pct"] / merged["n_occurrences"]).sum()

    # Drop unnecessary columns
    merged.drop(columns=["task_name", "task_normalized_temp", "pct"], inplace=True)

    # Reorder so `task` is first and `task_normalized` is second
    cols = ["task", "task_normalized"] + [c for c in merged.columns if c not in ["task", "task_normalized"]]
    merged = merged[cols]
    
    # Sort by O*NET-SOC Code
    merged.sort_values(by="soc_code_2010", ascending=True, inplace=True)

    return merged.reset_index(drop=True)


#### MCP Version

In [575]:
if mcp_run:

    # -----------------------------
    # Load base data
    # -----------------------------
    df = pd.read_csv(mcp_data)
    task_statements = pd.read_csv("../data/task_statements_v30.1.csv")
    crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # Winsorize n_ratings + pct
    # -----------------------------
    df["n_ratings"] = pd.to_numeric(df["n_ratings"], errors="coerce")

    q1 = df["n_ratings"].quantile(0.25)
    q3 = df["n_ratings"].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    df["n_ratings_capped"] = df["n_ratings"].clip(upper=upper_bound)
    df["pct"] = df["n_ratings_capped"] / df["n_ratings_capped"].sum()

    df = df[["occupation", "task", "pct"]].copy()

    # -----------------------------
    # Normalize merge keys
    # -----------------------------
    df["occupation_normalized"] = df["occupation"].apply(normalize_text)
    df["task_normalized"] = df["task"].apply(normalize_text)

    task_statements["occupation_normalized"] = task_statements["Title"].apply(normalize_text)
    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)

    # -----------------------------
    # Exact merge on occupation + task
    # -----------------------------
    df = df.merge(
        task_statements,
        on=["occupation_normalized", "task_normalized"],
        how="left"
    )

    # -----------------------------
    # Task substring fallback
    # -----------------------------
    task_lookup = task_statements.groupby("occupation_normalized")

    def fallback_match(row):
        if pd.notna(row["O*NET-SOC Code"]):
            return row["O*NET-SOC Code"]
        occ = row["occupation_normalized"]
        if occ not in task_lookup.groups:
            return None
        for _, cand in task_lookup.get_group(occ).iterrows():
            if row["task_normalized"] in cand["task_normalized"] or \
            cand["task_normalized"] in row["task_normalized"]:
                return cand["O*NET-SOC Code"]
        return None

    df["O*NET-SOC Code"] = df.apply(fallback_match, axis=1)

    # -----------------------------
    # Crosswalk 2019 → 2010
    # -----------------------------
    crosswalk = crosswalk.drop_duplicates(
        subset=["O*NET-SOC 2019 Code"],
        keep="first"
    )

    df = df.merge(
        crosswalk[[
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2010 Code",
            "O*NET-SOC 2010 Title"
        ]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal fallback
    df["soc_2019_base"] = df["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]

    crosswalk_base = crosswalk[[
        "soc_2019_base",
        "O*NET-SOC 2010 Code",
        "O*NET-SOC 2010 Title"
    ]].drop_duplicates(subset=["soc_2019_base"], keep="first")

    missing_mask = df["O*NET-SOC 2010 Code"].isna()

    fallback = df.loc[missing_mask, ["soc_2019_base"]].merge(
        crosswalk_base,
        on="soc_2019_base",
        how="left"
    )

    df.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # Final column formatting
    # -----------------------------
    df = df.rename(columns={
        "pct": "pct_normalized",
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source",
        "occupation": "title_current"
    })

    df["n_occurrences"] = (
        df.groupby("task_normalized")["title"]
        .transform("nunique")
    )

    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    df["pct_weighted"] = df["pct_weighted"] / df["pct_weighted"].sum()

    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized"
    ]].copy()

    df_finalized



#### Microsoft Data Version

In [576]:
if microsoft_run:

    # 1. Load and Filter iwa_metrics
    iwa_metrics = pd.read_csv(microsoft_data)
    iwa_metrics = iwa_metrics[iwa_metrics[['share_user', 'share_ai']].max(axis=1) >= 0.0005].copy()

    # 2. Load Tasks (Task-Occupation pairs)
    tasks_df = pd.read_csv("../data/tasks_dwa_iwa_gwa_v30.1_physical.csv")

    # Create normalized keys
    iwa_metrics['norm_title'] = iwa_metrics['title'].apply(normalize_text)
    tasks_df['norm_iwa_title'] = tasks_df['iwa_title'].apply(normalize_text)

    # 3. Calculate Weight per Task-Occupation Pair
    iwa_metrics['total_share'] = iwa_metrics[['share_user', 'share_ai']].sum(axis=1)
    iwa_metrics["iwa_title"] = iwa_metrics["title"]  # Keep original title for merging back later

    # The 'task_count' here is actually the count of Task-Occupation pairs per IWA
    iwa_pair_counts = tasks_df.groupby('norm_iwa_title').size().reset_index(name='pair_count')

    iwa_calc = pd.merge(iwa_metrics, iwa_pair_counts, left_on='norm_title', right_on='norm_iwa_title', how='left')
    iwa_calc['pair_pct_raw'] = iwa_calc['total_share'] / iwa_calc['pair_count']

    iwa_calc["impact_scope_ai_adj"] = iwa_calc["impact_scope_ai"] * (iwa_calc["share_ai"] / iwa_calc["total_share"])
    iwa_calc["impact_scope_user_adj"] = iwa_calc["impact_scope_user"] * (iwa_calc["share_user"] / iwa_calc["total_share"])

    # 4. Merge to distribute weights across all pairs
    # We keep 'title' (occupation) and 'task' to maintain the unique pairs
    merged_pairs = pd.merge(
        tasks_df[['Title', 'task', 'norm_iwa_title', "dwa_title", "iwa_title", "gwa_title"]], 
        iwa_calc[['norm_title', 'pair_pct_raw', 'impact_scope_ai', 'impact_scope_user', 
                  'impact_scope_ai_adj', 'impact_scope_user_adj']], 
        left_on='norm_iwa_title', 
        right_on='norm_title', 
        how='right' 
    )

    # 5. Global Normalization
    total_sum = merged_pairs['pair_pct_raw'].sum()
    merged_pairs['pct'] = merged_pairs['pair_pct_raw'] / total_sum

    # 6. Deduplicate Task-Occupation pairs just in case
    # We group by occupation (title) and task name
    # final_output = merged_pairs.drop_duplicates(subset=['Title', 'task']).copy()
    final_output = merged_pairs

    # Rename for final schema
    final_output = final_output.rename(columns={'task': 'task_name', 'Title': 'title_current'})

    # Final cleanup
    final_output = final_output[['title_current', 'task_name', 'pct', 'impact_scope_ai', 'impact_scope_user', 'impact_scope_ai_adj', 'impact_scope_user_adj', "dwa_title", "iwa_title", "gwa_title"]]

    print(f"Total Task-Occupation Pairs: {len(final_output)}")
    print(f"Pairs with valid pct: {final_output['pct'].notna().sum()}")

    final_output.to_csv("../data/task_pct_microsoft.csv", index=False)






    # -----------------------------
    # 1. Load Data
    # -----------------------------
    # This is the file we just created with [task_name, pct]
    # df = pd.read_csv("task_weights_by_iwa.csv") 
    df = final_output
    task_statements = pd.read_csv("../data/task_statements_v30.1.csv")
    crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # 2. Normalize and Merge
    # -----------------------------
    # We treat the 'pct' from the IWA file as our base 'pct_normalized'
    df = df.rename(columns={"task_name": "task", "pct": "pct_normalized"})

    # Create normalized keys for both Task and Occupation (Title)
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["title_normalized"] = df["title_current"].apply(normalize_text)

    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)
    task_statements["title_normalized"] = task_statements["Title"].apply(normalize_text)

    # Merge on BOTH task and occupation to get the correct O*NET metadata
    df = df.merge(
        task_statements,
        on=["task_normalized", "title_normalized"],
        how="left"
    )

    # -----------------------------
    # 3. Calculate Occurrences and Normalized Pct
    # -----------------------------
    # n_occurrences is how many occupations this task belongs to
    df["n_occurrences"] = (
        df.groupby("task_normalized")["title_current"]
        .transform("nunique")
    )

    # Per your instruction: divide pct by n_occurrences to get pct_normalized
    # Then re-normalize so the column sums to 1.0
    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    df["pct_weighted"] = df["pct_weighted"] / df["pct_weighted"].sum()

    # -----------------------------
    # 4. Crosswalk 2019 → 2010
    # -----------------------------
    crosswalk_clean = crosswalk.drop_duplicates(subset=["O*NET-SOC 2019 Code"], keep="first")

    df = df.merge(
        crosswalk_clean[[
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2010 Code",
            "O*NET-SOC 2010 Title"
        ]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal fallback for missing SOC codes
    df["soc_2019_base"] = df["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]
    crosswalk_base = crosswalk[["soc_2019_base", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"]].drop_duplicates("soc_2019_base")

    missing_mask = df["O*NET-SOC 2010 Code"].isna()
    fallback = df.loc[missing_mask, ["soc_2019_base"]].merge(crosswalk_base, on="soc_2019_base", how="left")

    df.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # 5. Final Column Formatting
    # -----------------------------
    df = df.rename(columns={
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source",
    })

    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized",
        'impact_scope_ai',
        'impact_scope_user',
        'impact_scope_ai_adj',
        'impact_scope_user_adj',
        "dwa_title",
        "iwa_title",
        "gwa_title"
    ]].copy()




#### Eco 2025

In [577]:
if eco_run_2025:

    DATA_DIR = Path("../data")


    # -----------------------------
    # 1. Load your Eco DF and O*NET base data
    # -----------------------------
    df_eco = pd.read_csv(DATA_DIR / "task_pct_eco_2025.csv") 
    task_statements = pd.read_csv(DATA_DIR / "task_statements_v30.1.csv")
    crosswalk = pd.read_csv(DATA_DIR / "2010_to_2019_soc_crosswalk.csv")

    # -----------------------------
    # 2. Normalize and Join
    # -----------------------------
    df_eco["task_normalized"] = df_eco["task_name"].apply(normalize_text)
    task_statements["task_normalized"] = task_statements["Task"].apply(normalize_text)

    # We include the extra O*NET columns here
    df_merged = df_eco.merge(
        task_statements[[
            'task_normalized', "Task", 'Title', 'O*NET-SOC Code', 'Task ID', 
            'Task Type', 'Incumbents Responding', 'Date', 'Domain Source'
        ]],
        on="task_normalized",
        how="left"
    )

    # -----------------------------
    # 3. Crosswalk 2019 -> 2010
    # -----------------------------
    crosswalk_clean = crosswalk.drop_duplicates(subset=["O*NET-SOC 2019 Code"], keep="first")

    df_merged = df_merged.merge(
        crosswalk_clean[["O*NET-SOC 2019 Code", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"]],
        left_on="O*NET-SOC Code",
        right_on="O*NET-SOC 2019 Code",
        how="left"
    )

    # Decimal Fallback Logic
    df_merged["soc_2019_base"] = df_merged["O*NET-SOC Code"].astype(str).str.split(".").str[0]
    crosswalk["soc_2019_base"] = crosswalk["O*NET-SOC 2019 Code"].astype(str).str.split(".").str[0]

    crosswalk_base = crosswalk[[
        "soc_2019_base", "O*NET-SOC 2010 Code", "O*NET-SOC 2010 Title"
    ]].drop_duplicates(subset=["soc_2019_base"], keep="first")

    missing_mask = df_merged["O*NET-SOC 2010 Code"].isna()
    fallback = df_merged.loc[missing_mask, ["soc_2019_base"]].merge(
        crosswalk_base, on="soc_2019_base", how="left"
    )

    df_merged.loc[missing_mask, "O*NET-SOC 2010 Code"] = fallback["O*NET-SOC 2010 Code"].values
    df_merged.loc[missing_mask, "O*NET-SOC 2010 Title"] = fallback["O*NET-SOC 2010 Title"].values

    # -----------------------------
    # 4. Rename & Weights (Mirroring the MCP logic)
    # -----------------------------
    df = df_merged.rename(columns={
        "Task": "task",
        "pct": "pct_normalized",
        "Title": "title_current",
        "O*NET-SOC Code": "soc_code_2019_full",
        "O*NET-SOC 2010 Code": "soc_code_2010",
        "O*NET-SOC 2010 Title": "title",
        "Task ID": "task_id",
        "Task Type": "task_type",
        "Incumbents Responding": "n_responding",
        "Date": "date",
        "Domain Source": "domain_source"
    })

    # Calculate how many Job Titles share each task
    df["n_occurrences"] = (
        df.groupby("task_normalized")["title"]
        .transform("nunique")
    )

    # Even though pct_normalized is 0, we perform the calculation to keep the column structure
    df["pct_weighted"] = df["pct_normalized"] * df["n_occurrences"]
    # If everything is zero, we handle the division by zero to keep it 0.0
    total_sum = df["pct_weighted"].sum()
    if total_sum > 0:
        df["pct_weighted"] = df["pct_weighted"] / total_sum
    else:
        df["pct_weighted"] = 0.0

    # -----------------------------
    # 5. Final Selection (Exact Columns)
    # -----------------------------
    df_finalized = df[[
        "task",
        "task_normalized",
        "soc_code_2019_full",
        "soc_code_2010",
        "title_current",
        "title",
        "task_id",
        "task_type",
        "n_responding",
        "date",
        "domain_source",
        "n_occurrences",
        "pct_weighted",
        "pct_normalized"
    ]].copy()

    output_path = DATA_DIR / "enriched_task_pct_eco_2025.csv"

    print(f"Success! Finalized data with {len(df_finalized)} rows saved to {output_path}")


Success! Finalized data with 18796 rows saved to ..\data\enriched_task_pct_eco_2025.csv


#### Run Function

In [578]:
if mcp_run or microsoft_run or eco_run_2025:
    pct_onet_tasks_df = df_finalized
elif eco_run_2015:
    task_statements_df = pd.read_csv("../data/task_statements_v20.1.csv")
    eco_data_2015 = pd.read_csv(eco_data_2015)
    pct_onet_tasks_df = pct_to_onet_tasks(eco_data_2015, task_statements_df)
else:
    task_statements_df = pd.read_csv("../data/task_statements_v20.1.csv")
    pct_df = pd.read_csv(pct_df)
    pct_onet_tasks_df = pct_to_onet_tasks(pct_df, task_statements_df)

if save_files_each_step:
    pct_onet_tasks_df.to_csv("../data/merged_data_files/pct_onet_tasks.csv", index=False)

## Step 2: Add SOC Major Occupational Category And Broad Counts

In [579]:
if not (mcp_run or microsoft_run or eco_run_2025):
    soc_crosswalk_df = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")
    soc_crosswalk_df = soc_crosswalk_df.rename(
        columns={
            "O*NET-SOC 2010 Title": "title",
            "O*NET-SOC 2019 Code": "onet_soc_code_2019"
        }
    )
    soc_crosswalk_df['soc_code_2019'] = soc_crosswalk_df['onet_soc_code_2019']
    # titles_df = pct_onet_tasks_df[["title", "broad_counts"]].drop_duplicates(subset=["title"])
    pct_onet_tasks_df = pct_onet_tasks_df.merge(
        soc_crosswalk_df[["title", "soc_code_2019"]],
        on="title",
        how="left"
    )
else:
    pct_onet_tasks_df["soc_code_2019"] = pct_onet_tasks_df["soc_code_2019_full"]



In [580]:
def add_soc_structure(pct_onet_tasks_df, soc_structure_df, detailed_occ_2019) -> pd.DataFrame:
    """
    Adds major_occ_category, minor_occ_category, broad_occ labels, and broad_counts
    to each row based on SOC structure data, with hard-coded fixes for known SOC anomalies.
    """

    soc_struct = soc_structure_df.copy()
    soc_struct.rename(columns={"SOC or O*NET-SOC 2019 Title": "soc_title"}, inplace=True)

    # ==========================
    # 🔧 FIX MINOR GROUP ERRORS
    # ==========================

    # Explicitly overwrite incorrect minor group codes
    soc_struct.loc[soc_struct["Minor Group"] == "15-1200", "Minor Group"] = "15-1000"
    soc_struct.loc[soc_struct["Minor Group"] == "31-1100", "Minor Group"] = "31-1000"
    soc_struct.loc[soc_struct["Minor Group"] == "51-5100", "Minor Group"] = "51-5000"

    # ==========================
    # BUILD TASK DF
    # ==========================

    df = pct_onet_tasks_df.copy()
    df["soc6"] = df["soc_code_2019"].astype(str).str[:7]
    df["major_group_code"] = df["soc6"].str[:2]
    df["minor_group_code"] = df["soc6"].str[:4] + "000"
    df["broad_code"] = df["soc6"].str[:-1] + "0"

    # ==========================================
    # 🔧 HARD-CODE BROAD FIX FOR 29-122x / 29-123x
    # ==========================================

    mask_29 = df["soc6"].str.startswith(("29-122", "29-123"))
    df.loc[mask_29, "broad_code"] = "29-1210"

    # ==========================
    # BUILD LOOKUPS
    # ==========================

    major_map = (
        soc_struct.loc[soc_struct["Major Group"].notna()]
        .drop_duplicates(subset=["Major Group"])
        .assign(key=lambda x: x["Major Group"].str[:2])
        .set_index("key")["soc_title"]
        .to_dict()
    )

    minor_map = (
        soc_struct.loc[soc_struct["Minor Group"].notna()]
        .drop_duplicates(subset=["Minor Group"])
        .set_index("Minor Group")["soc_title"]
        .to_dict()
    )

    broad_map = (
        soc_struct.loc[soc_struct["Broad Occupation"].notna()]
        .drop_duplicates(subset=["Broad Occupation"])
        .set_index("Broad Occupation")["soc_title"]
        .to_dict()
    )

    # ==========================
    # MAP LABELS
    # ==========================

    df["major_occ_category"] = df["major_group_code"].map(major_map)
    df["minor_occ_category"] = df["minor_group_code"].map(minor_map)
    df["broad_occ"] = df["broad_code"].map(broad_map)

    # ==========================
    # BROAD COUNTS
    # ==========================

    det = detailed_occ_2019.copy()
    det["soc6"] = det["O*NET-SOC 2019 Code"].str[:7]
    det["broad_code"] = det["soc6"].str[:-1] + "0"

    # Apply same 29 fix to counts
    mask_29_det = det["soc6"].str.startswith(("29-122", "29-123"))
    det.loc[mask_29_det, "broad_code"] = "29-1210"

    broad_counts_map = (
        det.groupby("broad_code")["soc6"]
        .nunique()
        .astype("Int64")
        .to_dict()
    )

    df["broad_counts"] = df["broad_code"].map(broad_counts_map)

    # Cleanup
    df.drop(columns=["soc6", "major_group_code", "minor_group_code", "broad_code", "soc_code_2019"], inplace=True)

    return df.reset_index(drop=True)

soc_structure_df = pd.read_csv("../data/soc_structure_2019.csv")
detailed_occ_2019 = pd.read_csv("../data/detailed_occ_2019.csv")
pct_tasks_soc_structure_df = add_soc_structure(pct_onet_tasks_df, soc_structure_df, detailed_occ_2019)

master_df = pd.read_csv("../data/master_pct_normalized.csv")

# Merge master_pct_normalized column onto the soc structure df
pct_tasks_soc_structure_df = pct_tasks_soc_structure_df.merge(
    master_df[['title', 'avg_total_pct_normalized']], 
    on='title', 
    how='left'
)

# Rename to your specific requested name
pct_tasks_soc_structure_df.rename(columns={'avg_total_pct_normalized': 'master_pct_normalized'}, inplace=True)

if save_files_each_step:
    pct_tasks_soc_structure_df.to_csv("../data/merged_data_files/pct_tasks_soc_structure.csv", index=False)



## Step 3: Add 2024 Wage and Employment Data

In [581]:
if mcp_run or microsoft_run or eco_run_2025:
    
    pct_tasks_soc_structure_df["soc_code_2019"] = (
        pct_tasks_soc_structure_df["soc_code_2019_full"]
        .astype(str)
        .str.split(".")
        .str[0]
    )

### 3.1: Add Updated (2019) SOC Codes

In [582]:
# Get df of updated SOC codes to merge with up to date wage and employment data

def add_updated_soc_code(pct_tasks_soc_structure_df, soc_crosswalk_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles from our main df and their corresponding O*NET-SOC 2019 code (some titles are duplicated as they get split into different SOC codes)
    This is so we can merge the wage and employment data separate from our main df and merge all at once. 

    Args:
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from previous step.
        soc_crosswalk_df (pd.DataFrame): DataFrame 2010 and 2019 occupation titles and SOC codes

    Returns:
        pd.DataFrame: DataFrame with an added 'soc_code_2019' column.
    """

    # Rename columns
    soc_crosswalk_df = soc_crosswalk_df.rename(
        columns={
            "O*NET-SOC 2010 Title": "title",
            "O*NET-SOC 2019 Code": "onet_soc_code_2019"
        }
    )

    soc_crosswalk_df['soc_code_2019'] = soc_crosswalk_df['onet_soc_code_2019'].str[:7]

    # Get unique titles from rolling DataFrame
    titles_df = pct_tasks_soc_structure_df[["title", "broad_counts"]].drop_duplicates(subset=["title"])

    # Merge to attach 2019 SOC codes
    merged = titles_df.merge(
        soc_crosswalk_df[["title", "soc_code_2019"]],
        on="title",
        how="left"
    )

    return merged


if mcp_run or microsoft_run or eco_run_2025:
    title_and_2019_soc_df = pct_tasks_soc_structure_df[["title", "soc_code_2019", "broad_counts"]]
else:
    soc_crosswalk_df = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")
    title_and_2019_soc_df = add_updated_soc_code(pct_tasks_soc_structure_df, soc_crosswalk_df)

if save_files_each_step:
    title_and_2019_soc_df.to_csv("../data/merged_data_files/title_and_2019_soc.csv", index=False)

In [583]:
title_and_2019_soc_df

,title,soc_code_2019,broad_counts
0,Chief Executives,11-1011,1
1,Chief Executives,11-1011,1
2,Chief Executives,11-1011,1
3,Chief Executives,11-1011,1
4,Chief Executives,11-1011,1
...,...,...,...
18791,"Tank Car, Truck, and Ship Loaders",53-7121,1
18792,"Tank Car, Truck, and Ship Loaders",53-7121,1
18793,"Tank Car, Truck, and Ship Loaders",53-7121,1
18794,"Tank Car, Truck, and Ship Loaders",53-7121,1


### 3.2: Add 2024 National Wage Data

In [584]:
def add_nat_wage_2024(title_and_2019_soc_df, nat_wage_df, scraped_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national annual and hourly median salary from 2024. 
    It also includes a 6 (from previous df) & 5 digit SOC code for use in following merging. 

    Args:
        title_and_2019_soc_df (pd.DataFrame): DataFrame from previous step.
        nat_wage_df (pd.DataFrame): DataFrame of OEWS data from 2024.
        scraped_wage_df (pd.DataFrame): DataFrame containing scraped wage data from O*NET's website from Jan 2020 

    Returns:
        pd.DataFrame: DataFrame with national wage data from 2024 added
    """

     # Get only columns needed
    wage_df_trimmed = nat_wage_df[["OCC_CODE", "O_GROUP", "H_MEDIAN", "A_MEDIAN"]].copy()
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

    # Change wage columns to floats
    for c in ["H_MEDIAN", "A_MEDIAN"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = title_and_2019_soc_df.merge(
        wage_df_trimmed, 
        on="soc_code_2019", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on
    merged["5_digit_soc"] = merged["soc_code_2019"].astype(str).str[:6]     
    wage_df_trimmed["5_digit_soc"] = wage_df_trimmed["soc_code_2019"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    wage_df_trimmed_fallback_1st = wage_df_trimmed[wage_df_trimmed["O_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["H_MEDIAN"].isna() | merged["A_MEDIAN"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        wage_df_trimmed_fallback_1st[["5_digit_soc", "H_MEDIAN", "A_MEDIAN"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "H_MEDIAN_fallback", "A_MEDIAN_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing median values from fallback columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEDIAN_fallback"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEDIAN_fallback"])

    # Create column to merge on and where annual median is missing
    scraped_wage_df["title"] = scraped_wage_df["JobName"]
    merged_fallback_2nd = merged[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].isna()]

    # Create 2nd fallback df with scraper wage data
    fallback_merge_2nd = merged_fallback_2nd.merge(
        scraped_wage_df[["title", "MedianSalary"]],
        on="title", how="left",
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_2nd_unique_titles = fallback_merge_2nd.drop_duplicates(subset="title")

    # Merge 2nd fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_2nd_unique_titles[["title", "MedianSalary"]],
        on="title",
        how="left"
    )

    # Fill missing median values from scraper median columns and make present value due to inflation
    inflation_factor = inflation_fac
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["MedianSalary"] * inflation_factor)

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["A_MEDIAN"].isna() & merged["H_MEDIAN"].notna(), "A_MEDIAN"] = (
        merged["H_MEDIAN"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].notna(), "H_MEDIAN"] = (
        merged["A_MEDIAN"] / 2080
    )

    # Create final national wage columns by averaging for any duplicate titles and drop uneeded columns. 
    merged["h_median_national"] = merged.groupby("title")["H_MEDIAN"].transform("mean")
    merged["a_median_national"] = merged.groupby("title")["A_MEDIAN"].transform("mean")
    merged.drop(columns=["H_MEDIAN", "A_MEDIAN", "H_MEDIAN_fallback", "A_MEDIAN_fallback", "MedianSalary", "O_GROUP"], inplace=True)

    return merged.reset_index(drop=True)


nat_wage_2024_df = pd.read_csv("../data/oews_national_2024.csv")
scraped_wage_df = pd.read_csv("../data/scraped_wage_data.csv")
titles_and_nat_wage_2024_df = add_nat_wage_2024(title_and_2019_soc_df, nat_wage_2024_df, scraped_wage_df, jan_2020_inflation_factor)

if save_files_each_step:
    titles_and_nat_wage_2024_df.to_csv("../data/merged_data_files/titles_and_nat_wage_2024.csv", index=False)


### 3.3: Add 2024 State Wage Data

In [585]:
# Build state name -> abbreviation mapping from the OEWS data
_state_df = pd.read_csv("../data/oews_states_2024.csv", usecols=["AREA_TITLE", "PRIM_STATE"])
STATE_ABBREV = (
    _state_df.drop_duplicates(subset="AREA_TITLE")
    .set_index("AREA_TITLE")["PRIM_STATE"]
    .str.lower()
    .to_dict()
)
ALL_STATES = list(STATE_ABBREV.keys())   # full names used for filtering
del _state_df
print(f"{len(ALL_STATES)} states/territories: {list(STATE_ABBREV.values())}")

54 states/territories: ['al', 'ak', 'az', 'ar', 'ca', 'co', 'ct', 'de', 'dc', 'fl', 'ga', 'hi', 'id', 'il', 'in', 'ia', 'ks', 'ky', 'la', 'me', 'md', 'ma', 'mi', 'mn', 'ms', 'mo', 'mt', 'ne', 'nv', 'nh', 'nj', 'nm', 'ny', 'nc', 'nd', 'oh', 'ok', 'or', 'pa', 'ri', 'sc', 'sd', 'tn', 'tx', 'ut', 'vt', 'va', 'wa', 'wv', 'wi', 'wy', 'gu', 'pr', 'vi']


In [586]:
def add_state_wage_2024(titles_and_nat_wage_df, state_wage_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state annual and hourly median salary from 2024
    for all states/territories.

    Args:
        titles_and_nat_wage_df (pd.DataFrame): DataFrame from previous step.
        state_wage_df (pd.DataFrame): DataFrame of OEWS data from 2024 with state level breakdown

    Returns:
        pd.DataFrame: DataFrame with state wage data from 2024 added (columns per state)
    """

    merged = titles_and_nat_wage_df.copy()

    for state_name, abbrev in STATE_ABBREV.items():
        # Get only columns needed, filtered to this state
        wage_df_trimmed = state_wage_df[["OCC_CODE", "H_MEDIAN", "A_MEDIAN", "AREA_TITLE"]].copy()
        wage_df_trimmed = wage_df_trimmed[wage_df_trimmed["AREA_TITLE"] == state_name]
        wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019",
                                        "H_MEDIAN": "h_median_state",
                                        "A_MEDIAN": "a_median_state"}, inplace=True)

        # Change wage columns to floats
        for c in ["h_median_state", "a_median_state"]:
            wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

        # Merge on detailed SOC codes
        merged = merged.merge(
            wage_df_trimmed,
            on="soc_code_2019",
            how="left"
        )

        # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
        merged.loc[merged["a_median_state"].isna() & merged["h_median_state"].notna(), "a_median_state"] = (
            merged["h_median_state"] * 2080
        )

        # Fill missing hourly median using annual median / 2080
        merged.loc[merged["h_median_state"].isna() & merged["a_median_state"].notna(), "h_median_state"] = (
            merged["a_median_state"] / 2080
        )

        # Fill remaining missing values with national data
        merged.loc[merged["a_median_state"].isna(), "a_median_state"] = merged["a_median_national"]
        merged.loc[merged["h_median_state"].isna(), "h_median_state"] = merged["h_median_national"]

        # Aggregate to title level and store as named columns
        merged[f"h_median_{abbrev}"] = merged.groupby("title")["h_median_state"].transform("mean")
        merged[f"a_median_{abbrev}"] = merged.groupby("title")["a_median_state"].transform("mean")
        merged.drop(columns=["h_median_state", "a_median_state", "AREA_TITLE"], inplace=True)

    return merged


state_wage_df_2024 = pd.read_csv("../data/oews_states_2024.csv")
titles_nat_and_state_wage_2024_df = add_state_wage_2024(titles_and_nat_wage_2024_df, state_wage_df_2024)

if save_files_each_step:
    titles_nat_and_state_wage_2024_df.to_csv("../data/merged_data_files/titles_nat_and_state_wage_2024.csv", index=False)


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3145390723.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"a_median_{abbrev}"] = merged.groupby("title")["a_median_state"].transform("mean")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3145390723.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"h_median_{abbrev}"] = merged.groupby("title")["h_median_state"].transform("mean")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3145390723.py:51: PerformanceWarning: DataFrame is highly fragmented.  This i

### 3.4: Add 2024 National Employment Data

In [587]:
def add_nat_emp_2024(titles_nat_and_state_wage_df, nat_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national employment data from 2024.  

    Args:
        titles_nat_and_state_wage_df (pd.DataFrame): DataFrame from previous step.
        nat_emp_df (pd.DataFrame): DataFrame of OEWS data from 2024.

    Returns:
        pd.DataFrame: DataFrame with national employment data from 2024 added
    """

     # Get only columns needed
    emp_df_trimmed = nat_emp_df[["OCC_CODE", "TOT_EMP", "O_GROUP"]].copy()
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

    # Change emp columns to floats
    emp_df_trimmed["TOT_EMP"] = pd.to_numeric(emp_df_trimmed["TOT_EMP"], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_nat_and_state_wage_df.merge(
        emp_df_trimmed, 
        on="soc_code_2019", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on  
    emp_df_trimmed["5_digit_soc"] = emp_df_trimmed["soc_code_2019"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    emp_df_trimmed_fallback_1st = emp_df_trimmed[emp_df_trimmed["O_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["TOT_EMP"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    fallback_merge["TOT_EMP_fallback"] = fallback_merge["TOT_EMP_fallback"] / fallback_merge["broad_counts"]

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "TOT_EMP_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing emp values from fallback columns
    merged["TOT_EMP"] = merged["TOT_EMP"].fillna(merged["TOT_EMP_fallback"])

    # After the initial merge, count how many titles share each SOC code
    soc_title_counts = merged.groupby("soc_code_2019")["title"].transform("nunique")

    # Divide employment by number of titles sharing that SOC code
    merged["TOT_EMP"] = merged["TOT_EMP"] / soc_title_counts

    # Create final national emp columns by dividing by number of occurences for each soc code and summing per occupation. 
    title_counts = merged.groupby("title")["soc_code_2019"].transform("count")
    merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
    merged["emp_total_national"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")

    merged.drop(columns=["TOT_EMP_fallback", "TOT_EMP", "O_GROUP", "TOT_EMP_adj", "broad_counts"], inplace=True)
    return merged.reset_index(drop=True)


nat_emp_df_2024 = pd.read_csv("../data/oews_national_2024.csv")
titles_wage_nat_emp_2024_df = add_nat_emp_2024(titles_nat_and_state_wage_2024_df, nat_emp_df_2024)

if save_files_each_step:
    titles_wage_nat_emp_2024_df.to_csv("../data/merged_data_files/titles_wage_nat_emp_2024.csv", index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3075261197.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3075261197.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["emp_total_national"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")


### 3.5: Add 2024 State Employment Data

In [588]:
def add_state_emp_2024(titles_wage_nat_emp_df, state_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state employment data from 2024
    for all states/territories.

    Args:
        titles_wage_nat_emp_df (pd.DataFrame): DataFrame from previous step.
        state_emp_df (pd.DataFrame): DataFrame of OEWS data from 2024.

    Returns:
        pd.DataFrame: DataFrame with state employment data from 2024 added (columns per state)
    """

    # Change emp column to float once
    state_emp_df["TOT_EMP"] = pd.to_numeric(state_emp_df["TOT_EMP"], errors="coerce")

    merged = titles_wage_nat_emp_df.copy()

    for state_name, abbrev in STATE_ABBREV.items():
        # Get only columns needed, filtered to this state
        emp_df_trimmed = state_emp_df[["OCC_CODE", "TOT_EMP", "AREA_TITLE"]].copy()
        emp_df_trimmed = emp_df_trimmed[emp_df_trimmed["AREA_TITLE"] == state_name]
        emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2019"}, inplace=True)

        # Merge on detailed SOC codes
        merged = merged.merge(
            emp_df_trimmed,
            on="soc_code_2019",
            how="left"
        )

        # Fill missing values with national data * state share of employment
        total_nat_emp = state_emp_df.loc[state_emp_df["OCC_CODE"] == "00-0000", "TOT_EMP"].sum()
        total_state_emp = state_emp_df.loc[
            (state_emp_df["OCC_CODE"] == "00-0000") & (state_emp_df["AREA_TITLE"] == state_name), "TOT_EMP"
        ]
        if total_state_emp.empty:
            state_share = 0.0
        else:
            state_share = float(total_state_emp.iloc[0]) / float(total_nat_emp)
        merged.loc[merged["TOT_EMP"].isna(), "TOT_EMP"] = (
            (merged["emp_total_national"] * state_share).round()
        )

        # Aggregate to title level
        title_counts = merged.groupby("title")["soc_code_2019"].transform("count")
        merged["TOT_EMP_adj"] = merged["TOT_EMP"] / title_counts
        merged[f"emp_total_{abbrev}"] = merged.groupby("title")["TOT_EMP_adj"].transform("sum")

        merged.drop(columns=["TOT_EMP", "AREA_TITLE", "TOT_EMP_adj"], inplace=True)

    return merged.reset_index(drop=True)


state_emp_2024_df = pd.read_csv("../data/oews_states_2024.csv")
titles_wage_all_emp_2024_df = add_state_emp_2024(titles_wage_nat_emp_2024_df, state_emp_2024_df)

if save_files_each_step:
    titles_wage_all_emp_2024_df.to_csv("../data/merged_data_files/titles_wage_all_emp_2024.csv", index=False)

In [589]:
titles_wage_all_emp_2024_df

,title,soc_code_2019,5_digit_soc,h_median_national,a_median_national,h_median_al,a_median_al,h_median_ak,a_median_ak,h_median_az,...,emp_total_ut,emp_total_vt,emp_total_va,emp_total_wa,emp_total_wv,emp_total_wi,emp_total_wy,emp_total_gu,emp_total_pr,emp_total_vi
0,Chief Executives,11-1011,11-101,99.24,206420.0,79.04,164400.0,81.10,168680.0,72.40,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
1,Chief Executives,11-1011,11-101,99.24,206420.0,79.04,164400.0,81.10,168680.0,72.40,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
2,Chief Executives,11-1011,11-101,99.24,206420.0,79.04,164400.0,81.10,168680.0,72.40,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
3,Chief Executives,11-1011,11-101,99.24,206420.0,79.04,164400.0,81.10,168680.0,72.40,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
4,Chief Executives,11-1011,11-101,99.24,206420.0,79.04,164400.0,81.10,168680.0,72.40,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18791,"Tank Car, Truck, and Ship Loaders",53-7121,53-712,27.92,58070.0,40.99,85260.0,40.06,83330.0,16.88,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18792,"Tank Car, Truck, and Ship Loaders",53-7121,53-712,27.92,58070.0,40.99,85260.0,40.06,83330.0,16.88,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18793,"Tank Car, Truck, and Ship Loaders",53-7121,53-712,27.92,58070.0,40.99,85260.0,40.06,83330.0,16.88,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18794,"Tank Car, Truck, and Ship Loaders",53-7121,53-712,27.92,58070.0,40.99,85260.0,40.06,83330.0,16.88,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0


### 3.6: Merge 2024 Wage and Employment Data Into Task Data

In [590]:
def wage_emp_to_tasks_2024(titles_wage_all_emp_df, pct_tasks_soc_structure_df) -> pd.DataFrame:
    """
    Returns DataFrame with our wage and employment data from 2024 added to our task data.  

    Args:
        titles_wage_all_emp_df (pd.DataFrame): DataFrame from previous step.
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from step 2

    Returns:
        pd.DataFrame: DataFrame with wage and employment data from 2024 added to task data
    """

    titles_wage_all_emp_df = titles_wage_all_emp_df.drop_duplicates(subset="title").copy()

    titles_wage_all_emp_df.drop(columns=["5_digit_soc", "soc_code_2019"], inplace=True)

    merged = pct_tasks_soc_structure_df.merge(
        titles_wage_all_emp_df,
        on="title",
        how="left"
    )

    # Build rename dict: national + all states
    rename_map = {
        "h_median_national": "h_med_nat_2024",
        "a_median_national": "a_med_nat_2024",
        "emp_total_national": "emp_tot_nat_2024",
    }
    for abbrev in STATE_ABBREV.values():
        rename_map[f"h_median_{abbrev}"] = f"h_med_{abbrev}_2024"
        rename_map[f"a_median_{abbrev}"] = f"a_med_{abbrev}_2024"
        rename_map[f"emp_total_{abbrev}"] = f"emp_tot_{abbrev}_2024"

    merged.rename(columns=rename_map, inplace=True)
    
    return merged


task_wage_emp_2024_df = wage_emp_to_tasks_2024(titles_wage_all_emp_2024_df, pct_tasks_soc_structure_df)

if save_files_each_step:
    task_wage_emp_2024_df.to_csv("../data/merged_data_files/task_wage_emp_2024.csv", index=False)

In [591]:
task_wage_emp_2024_df

,task,task_normalized,soc_code_2019_full,soc_code_2010,title_current,title,task_id,task_type,n_responding,date,...,emp_tot_ut_2024,emp_tot_vt_2024,emp_tot_va_2024,emp_tot_wa_2024,emp_tot_wv_2024,emp_tot_wi_2024,emp_tot_wy_2024,emp_tot_gu_2024,emp_tot_pr_2024,emp_tot_vi_2024
0,Direct or coordinate an organization's financi...,direct or coordinate an organizations financia...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8823,Core,95.0,08/2023,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8824,Core,95.0,08/2023,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
2,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8827,Core,95.0,08/2023,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
3,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8826,Core,94.0,08/2023,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
4,Prepare or present reports concerning activiti...,prepare or present reports concerning activiti...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8834,Core,95.0,08/2023,...,3980.0,530.0,4620.0,4140.0,1390.0,4440.0,70.0,320.0,2340.0,120.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18791,Unload cars containing liquids by connecting h...,unload cars containing liquids by connecting h...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12807,Supplemental,85.0,08/2019,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18792,"Clean interiors of tank cars or tank trucks, u...",clean interiors of tank cars or tank trucks us...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12804,Supplemental,85.0,08/2019,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18793,Lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12803,Supplemental,85.0,08/2019,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0
18794,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12805,Supplemental,85.0,08/2019,...,40.0,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0


## Step 4: Add 2015 Wage and Employment Data

### 4.1: Add 2015 National Wage Data

In [592]:
def add_nat_wage_2015(pct_tasks_soc_structure_df, nat_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Creates a DataFrame of titles and their 2010 SOC codes
    Returns DataFrame with occupation titles along with their national annual and hourly median salary from 2015 in real and nominal terms merged with titles and SOC codes. 
    It also includes a 5 digit SOC code for use in following merging. 

    Args:
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from Step 2.
        nat_wage_df (pd.DataFrame): DataFrame of OEWS data from 2015 

    Returns:
        pd.DataFrame: DataFrame with national wage data from 2024 added
    """

    # Make df with titles and SOC codes
    title_soc_code_2010_df = pct_tasks_soc_structure_df[["title", "soc_code_2010", "broad_counts"]].drop_duplicates(subset="title").copy()
    title_soc_code_2010_df.reset_index(drop=True, inplace=True)
    title_soc_code_2010_df['soc_code_2010'] = title_soc_code_2010_df['soc_code_2010'].str[:7]

    # Get only columns needed
    wage_df_trimmed = nat_wage_df[["OCC_CODE", "OCC_GROUP", "H_MEDIAN", "A_MEDIAN", "H_MEAN", "A_MEAN"]].copy()
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Change wage columns to floats
    for c in ["H_MEDIAN", "A_MEDIAN", "H_MEAN", "A_MEAN"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = title_soc_code_2010_df.merge(
        wage_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["A_MEDIAN"].isna() & merged["H_MEDIAN"].notna(), "A_MEDIAN"] = (
        merged["H_MEDIAN"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["H_MEDIAN"].isna() & merged["A_MEDIAN"].notna(), "H_MEDIAN"] = (
        merged["A_MEDIAN"] / 2080
    )

    # Get 5 digit SOC codes for broad groups to merge on
    merged["5_digit_soc"] = merged["soc_code_2010"].astype(str).str[:6]     
    wage_df_trimmed["5_digit_soc"] = wage_df_trimmed["soc_code_2010"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    wage_df_trimmed_fallback_1st = wage_df_trimmed[wage_df_trimmed["OCC_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["H_MEDIAN"].isna() | merged["A_MEDIAN"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        wage_df_trimmed_fallback_1st[["5_digit_soc", "H_MEDIAN", "A_MEDIAN"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "H_MEDIAN_fallback", "A_MEDIAN_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing median values from fallback columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEDIAN_fallback"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEDIAN_fallback"])

    # Fill missing median values from mean columns
    merged["H_MEDIAN"] = merged["H_MEDIAN"].fillna(merged["H_MEAN"])
    merged["A_MEDIAN"] = merged["A_MEDIAN"].fillna(merged["A_MEAN"])

    # Rename and drop columns for cleanup 
    merged.rename(columns={"H_MEDIAN": "h_med_nat_nominal"}, inplace=True)
    merged.rename(columns={"A_MEDIAN": "a_med_nat_nominal"}, inplace=True)
    merged.drop(columns=["H_MEDIAN_fallback", "A_MEDIAN_fallback", "H_MEAN", "A_MEAN", "OCC_GROUP"], inplace=True)

    # Make present value column for inflation
    inflation_factor = inflation_fac
    merged["h_med_nat_real"] = merged["h_med_nat_nominal"] * inflation_factor
    merged["a_med_nat_real"] = merged["a_med_nat_nominal"] * inflation_factor

    return merged.reset_index(drop=True)


nat_wage_df_2015 = pd.read_csv("../data/oews_national_2015.csv")
titles_and_nat_wage_2015_df = add_nat_wage_2015(pct_tasks_soc_structure_df, nat_wage_df_2015, may_2015_inflation_factor)

if save_files_each_step:
    titles_and_nat_wage_2015_df.to_csv("../data/merged_data_files/titles_and_nat_wage_2015.csv", index=False)

### 4.2: Add 2015 State Wage Data

In [593]:
def add_state_wage_2015(titles_and_nat_wage_df, state_wage_df, inflation_fac) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state annual and hourly median salary from 2015 in nominal and real terms. 

    Args:
        titles_and_nat_wage_df (pd.DataFrame): DataFrame from previous step.
        state_wage_df (pd.DataFrame): DataFrame of OEWS data from 2015 with state level breakdown

    Returns:
        pd.DataFrame: DataFrame with state wage data from 2015 added
    """

    # Get only columns needed
    wage_df_trimmed = state_wage_df[["OCC_CODE", "H_MEDIAN", "A_MEDIAN", "ST"]].copy()
    wage_df_trimmed = wage_df_trimmed[wage_df_trimmed["ST"] == "UT"]
    wage_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010",
                                    "H_MEDIAN": "h_median_state",
                                    "A_MEDIAN": "a_median_state"}, inplace=True)

    # Change wage columns to floats
    for c in ["h_median_state", "a_median_state"]:
        wage_df_trimmed[c] = pd.to_numeric(wage_df_trimmed[c], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_and_nat_wage_df.merge(
        wage_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill missing annual median using hourly median * 2080 (52 weeks * 40 hours)
    merged.loc[merged["a_median_state"].isna() & merged["h_median_state"].notna(), "a_median_state"] = (
        merged["h_median_state"] * 2080
    )

    # Fill missing hourly median using annual median / 2080
    merged.loc[merged["h_median_state"].isna() & merged["a_median_state"].notna(), "h_median_state"] = (
        merged["a_median_state"] / 2080
    )

    # Fill remaining missing values with national data
    merged.loc[merged["a_median_state"].isna(), "a_median_state"] = (
        merged["a_med_nat_nominal"]
    )
    merged.loc[merged["h_median_state"].isna(), "h_median_state"] = (
        merged["h_med_nat_nominal"]
    )

    # Rename and drop columns for cleanup
    merged.rename(columns={"h_median_state": "h_med_utah_nominal",
                                    "a_median_state": "a_med_utah_nominal"}, inplace=True)
    merged.drop(columns=["ST"], inplace=True)

    # Make present value column for inflation
    inflation_factor = inflation_fac
    merged["h_med_utah_real"] = merged["h_med_utah_nominal"] * inflation_factor
    merged["a_med_utah_real"] = merged["a_med_utah_nominal"] * inflation_factor

    return merged.reset_index(drop=True)


state_wage_df_2015 = pd.read_csv("../data/oews_states_2015.csv")
titles_nat_and_state_wage_2015_df = add_state_wage_2015(titles_and_nat_wage_2015_df, state_wage_df_2015, may_2015_inflation_factor)

if save_files_each_step:
    titles_nat_and_state_wage_2015_df.to_csv("../data/merged_data_files/titles_nat_and_state_wage_2015.csv", index=False)

### 4.3: Add 2015 National Employment Data

In [594]:
def add_nat_emp_2015(titles_nat_and_state_wage_df, nat_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their national employment data from 2015.  

    Args:
        titles_nat_and_state_wage_df (pd.DataFrame): DataFrame from previous step.
        nat_emp_df (pd.DataFrame): DataFrame of OEWS data from 2015.

    Returns:
        pd.DataFrame: DataFrame with national employment data from 2015 added
    """

    # Get only columns needed
    emp_df_trimmed = nat_emp_df[["OCC_CODE", "TOT_EMP", "OCC_GROUP"]].copy()
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Change emp columns to floats
    emp_df_trimmed["TOT_EMP"] = pd.to_numeric(emp_df_trimmed["TOT_EMP"], errors="coerce")

    # Initial merge on detailed SOC codes
    merged = titles_nat_and_state_wage_df.merge(
        emp_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Get 5 digit SOC codes for broad groups to merge on  
    emp_df_trimmed["5_digit_soc"] = emp_df_trimmed["soc_code_2010"].astype(str).str[:6]

    #Create fallback DataFrames with only broad groups and where median values are missing
    emp_df_trimmed_fallback_1st = emp_df_trimmed[emp_df_trimmed["OCC_GROUP"] == "broad"]
    merged_fallback_1st = merged[merged["TOT_EMP"].isna()]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    fallback_merge["TOT_EMP_fallback"] = fallback_merge["TOT_EMP_fallback"] / fallback_merge["broad_counts"]

    # Create fallback df with broad group wages
    fallback_merge = merged_fallback_1st.merge(
        emp_df_trimmed_fallback_1st[["5_digit_soc", "TOT_EMP"]],
        on="5_digit_soc", how="left",
        suffixes=("", "_fallback")
    )

    # Make titles unique so we don't create a Cartesian product when merging into main DataFrame
    fallback_merge_unique_titles = fallback_merge.drop_duplicates(subset="title")

    # Merge fallback data into the main dataframe
    merged = merged.merge(
        fallback_merge_unique_titles[["title", "TOT_EMP_fallback"]],
        on="title",
        how="left"
    )

    # Fill missing emp values from fallback columns
    merged["TOT_EMP"] = merged["TOT_EMP"].fillna(merged["TOT_EMP_fallback"])

    # Rename and drop columns for cleanup
    merged.rename(columns={"TOT_EMP": "emp_tot_nat"}, inplace=True)
    merged.drop(columns=["TOT_EMP_fallback", "OCC_GROUP", "broad_counts"], inplace=True)

    return merged.reset_index(drop=True)


nat_emp_df_2015 = pd.read_csv("../data/oews_national_2015.csv")
titles_wage_nat_emp_2015_df = add_nat_emp_2015(titles_nat_and_state_wage_2015_df, nat_emp_df_2015)

if save_files_each_step:
    titles_wage_nat_emp_2015_df.to_csv("../data/merged_data_files/titles_wage_nat_emp_2015.csv", index=False)

### 4.4: Add 2015 State Employment Data

In [595]:
def add_state_emp_2015(titles_wage_nat_emp_df, state_emp_df) -> pd.DataFrame:
    """
    Returns DataFrame with occupation titles along with their state employment data from 2015.  

    Args:
        titles_wage_nat_emp_df (pd.DataFrame): DataFrame from previous step.
        state_emp_df (pd.DataFrame): DataFrame of OEWS data from 2015.

    Returns:
        pd.DataFrame: DataFrame with state employment data from 2015 added
    """

    # Change emp columns to floats
    state_emp_df["TOT_EMP"] = pd.to_numeric(state_emp_df["TOT_EMP"], errors="coerce")

    # Get only columns needed
    emp_df_trimmed = state_emp_df[["OCC_CODE", "TOT_EMP", "ST"]].copy()
    emp_df_trimmed = emp_df_trimmed[emp_df_trimmed["ST"] == "UT"]
    emp_df_trimmed.rename(columns={"OCC_CODE": "soc_code_2010"}, inplace=True)

    # Initial merge on detailed SOC codes
    merged = titles_wage_nat_emp_2015_df.merge(
        emp_df_trimmed, 
        on="soc_code_2010", 
        how="left"
    )

    # Fill remaining missing values with national data by multiplying by the proportion of state employment to national employment
    total_nat_emp = state_emp_df.loc[state_emp_df["OCC_CODE"] == "00-0000", "TOT_EMP"].sum()
    total_utah_emp = state_emp_df.loc[(state_emp_df["OCC_CODE"] == "00-0000") & (state_emp_df["ST"] == "UT"), "TOT_EMP"].iloc[0]
    utah_share = float(total_utah_emp) / float(total_nat_emp)
    merged.loc[merged["TOT_EMP"].isna(), "TOT_EMP"] = (
    (merged["emp_tot_nat"] * utah_share).round())

    # Rename and drop columns for cleanup
    merged.rename(columns={"TOT_EMP": "emp_tot_utah"}, inplace=True)
    merged.drop(columns=["ST"], inplace=True)

    return merged.reset_index(drop=True)


state_emp_2015_df = pd.read_csv("../data/oews_states_2015.csv")
titles_wage_all_emp_2015_df = add_state_emp_2015(titles_wage_nat_emp_2015_df, state_emp_2015_df)

if save_files_each_step:
    titles_wage_all_emp_2015_df.to_csv("../data/merged_data_files/titles_wage_all_emp_2015.csv", index=False)

In [596]:
titles_wage_all_emp_2015_df

,title,soc_code_2010,h_med_nat_nominal,a_med_nat_nominal,5_digit_soc,h_med_nat_real,a_med_nat_real,h_med_utah_nominal,a_med_utah_nominal,h_med_utah_real,a_med_utah_real,emp_tot_nat,emp_tot_utah
0,Chief Executives,11-1011,84.190000,175110.0,11-101,114.498400,238149.6,62.140000,129260.0,84.510400,175793.6,238940.0,3770.0
1,Chief Sustainability Officers,11-1011,84.190000,175110.0,11-101,114.498400,238149.6,62.140000,129260.0,84.510400,175793.6,238940.0,3770.0
2,General and Operations Managers,11-1021,46.990000,97730.0,11-102,63.906400,132912.8,33.720000,70150.0,45.859200,95404.0,2145140.0,31180.0
3,Legislators,11-1031,9.855769,20500.0,11-103,13.403846,27880.0,9.235577,19210.0,12.560385,26125.6,55820.0,670.0
4,Advertising and Promotions Managers,11-2011,46.100000,95890.0,11-201,62.696000,130410.4,35.580000,74010.0,48.388800,100653.6,29340.0,330.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,Gas Compressor and Gas Pumping Station Operators,53-7071,28.060000,58350.0,53-707,38.161600,79356.0,26.260000,54630.0,35.713600,74296.8,4100.0,90.0
894,"Pump Operators, Except Wellhead Pumpers",53-7072,20.390000,42420.0,53-707,27.730400,57691.2,20.390000,42420.0,27.730400,57691.2,13390.0,128.0
895,Wellhead Pumpers,53-7073,22.590000,46990.0,53-707,30.722400,63906.4,26.380000,54870.0,35.876800,74623.2,12860.0,120.0
896,Refuse and Recyclable Material Collectors,53-7081,16.250000,33800.0,53-708,22.100000,45968.0,16.190000,33680.0,22.018400,45804.8,114220.0,770.0


### 4.5: Merge 2015 Wage and Employment Data Into Task Data

In [597]:
def wage_emp_to_tasks_2015(titles_wage_all_emp_df, pct_tasks_soc_structure_df) -> pd.DataFrame:
    """
    Returns DataFrame with our wage and employment data from 2015 added to our task data.  

    Args:
        titles_wage_all_emp_df (pd.DataFrame): DataFrame from previous step.
        pct_tasks_soc_structure_df (pd.DataFrame): DataFrame from step 2

    Returns:
        pd.DataFrame: DataFrame with wage and employment data from 2015 added to task data
    """

    titles_wage_all_emp_df = titles_wage_all_emp_df.drop_duplicates(subset="title")

    titles_wage_all_emp_df.drop(columns=["soc_code_2010", "5_digit_soc"], inplace=True)

    merged = pct_tasks_soc_structure_df.merge(
        titles_wage_all_emp_df,
        on="title",
        how="left"
    )

    merged.rename(columns={"h_med_nat_nominal": "h_med_nat_nominal_2015",
                            "a_med_nat_nominal": "a_med_nat_nominal_2015",
                            "h_med_nat_real": "h_med_nat_real_2015",
                            "a_med_nat_real": "a_med_nat_real_2015",
                            "h_med_utah_nominal": "h_med_ut_nominal_2015",
                            "a_med_utah_nominal": "a_med_ut_nominal_2015",
                            "h_med_utah_real": "h_med_ut_real_2015",
                            "a_med_utah_real": "a_med_ut_real_2015",
                            "emp_tot_nat": "emp_tot_nat_2015",
                            "emp_tot_utah": "emp_tot_ut_2015"}, inplace=True)
    
    return merged
    

tasks_all_wage_emp_df = wage_emp_to_tasks_2015(titles_wage_all_emp_2015_df, task_wage_emp_2024_df)

if save_files_each_step:
    tasks_all_wage_emp_df.to_csv("../data/merged_data_files/tasks_all_wage_emp.csv", index=False)

In [598]:
tasks_all_wage_emp_df

,task,task_normalized,soc_code_2019_full,soc_code_2010,title_current,title,task_id,task_type,n_responding,date,...,h_med_nat_nominal_2015,a_med_nat_nominal_2015,h_med_nat_real_2015,a_med_nat_real_2015,h_med_ut_nominal_2015,a_med_ut_nominal_2015,h_med_ut_real_2015,a_med_ut_real_2015,emp_tot_nat_2015,emp_tot_ut_2015
0,Direct or coordinate an organization's financi...,direct or coordinate an organizations financia...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8823,Core,95.0,08/2023,...,84.19,175110.0,114.4984,238149.6,62.14,129260.0,84.5104,175793.6,238940.0,3770.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8824,Core,95.0,08/2023,...,84.19,175110.0,114.4984,238149.6,62.14,129260.0,84.5104,175793.6,238940.0,3770.0
2,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8827,Core,95.0,08/2023,...,84.19,175110.0,114.4984,238149.6,62.14,129260.0,84.5104,175793.6,238940.0,3770.0
3,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8826,Core,94.0,08/2023,...,84.19,175110.0,114.4984,238149.6,62.14,129260.0,84.5104,175793.6,238940.0,3770.0
4,Prepare or present reports concerning activiti...,prepare or present reports concerning activiti...,11-1011.00,11-1011.00,Chief Executives,Chief Executives,8834,Core,95.0,08/2023,...,84.19,175110.0,114.4984,238149.6,62.14,129260.0,84.5104,175793.6,238940.0,3770.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18791,Unload cars containing liquids by connecting h...,unload cars containing liquids by connecting h...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12807,Supplemental,85.0,08/2019,...,17.63,36660.0,23.9768,49857.6,17.63,36660.0,23.9768,49857.6,11960.0,114.0
18792,"Clean interiors of tank cars or tank trucks, u...",clean interiors of tank cars or tank trucks us...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12804,Supplemental,85.0,08/2019,...,17.63,36660.0,23.9768,49857.6,17.63,36660.0,23.9768,49857.6,11960.0,114.0
18793,Lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12803,Supplemental,85.0,08/2019,...,17.63,36660.0,23.9768,49857.6,17.63,36660.0,23.9768,49857.6,11960.0,114.0
18794,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,53-7121.00,53-7121.00,"Tank Car, Truck, and Ship Loaders","Tank Car, Truck, and Ship Loaders",12805,Supplemental,85.0,08/2019,...,17.63,36660.0,23.9768,49857.6,17.63,36660.0,23.9768,49857.6,11960.0,114.0


## Step 5: Adjust Employment Columns

In [599]:
def adjust_emp_old(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2024_df, nat_emp_df_2015, nat_emp_df_2024) -> pd.DataFrame:
    """
    Reallocates employment numbers based on the relative percent of Claude conversations, as we have some duplicate
    6 digit SOC codes but different titles  

    Args:
        tasks_all_wage_emp_df (pd.DataFrame): DataFrame from previous 4.5.

    Returns:
        pd.DataFrame: DataFrame with correct employment numbers
    """

    df = tasks_all_wage_emp_df

    # 6-digit SOC to remove decimals (e.g., '11-1011.03' -> '11-1011')
    df["soc6"] = df["soc_code_2010"].astype(str).str[:7]

    # 1. Fill NaNs with the group average (or 0 if the whole group is NaN)
    df["master_pct_normalized"] = df.groupby("soc6")["master_pct_normalized"].transform(lambda x: x.fillna(x.mean())).fillna(0)

    # 2. Dampen extreme values (Square Root) to bring values closer to the mean
    # This prevents outliers from "eating" all the employment in a SOC6 group
    df["master_pct_adj"] = np.sqrt(df["master_pct_normalized"])

    # 3. Calculate shares with a fallback to equal split
    title_val = df.groupby(["soc6", "title"])["master_pct_adj"].transform("first")
    soc6_total = df.groupby("soc6")["master_pct_adj"].transform(lambda x: x.unique().sum())
    title_counts = df.groupby("soc6")["title"].transform("nunique")

    # If the group total is 0, split equally. Otherwise, use the dampened shares.
    df["soc6_share"] = np.where(
        soc6_total > 0,
        title_val / soc6_total,
        1 / title_counts
    )

    # columns to allocate � national + all states for both years
    emp_cols = [c for c in df.columns if c.startswith("emp_tot_") and c.endswith(("_2024", "_2015"))]

    # For each employment column, check if reallocation is needed
    for c in emp_cols:
        # Get one employment value per title within each soc6
        emp_per_title = df.groupby(["soc6", "title"])[c].first().reset_index()
        
        # For each soc6, check if all titles have the same employment (duplicates from fallback)
        def has_duplicate_emp(group):
            return (group[c].nunique() == 1) and (len(group) > 1)
        
        soc6_needs_adjustment = emp_per_title.groupby("soc6").apply(has_duplicate_emp, include_groups=False)
        soc6_to_adjust = soc6_needs_adjustment[soc6_needs_adjustment].index.tolist()
        
        # Create mask for rows that need adjustment
        needs_adjustment = df["soc6"].isin(soc6_to_adjust)
        
        # Get the soc6 total by SUMMING across titles (not max!)
        emp_sum_per_title = df.groupby(["soc6", "title"])[c].first()
        soc6_sum = emp_sum_per_title.groupby("soc6").sum()
        soc6_tot_corrected = soc6_sum 
        soc6_tot = df["soc6"].map(soc6_tot_corrected)  # Map back to dataframe
        
        # Only reallocate where needed, otherwise keep original
        df[c] = np.where(
            needs_adjustment,
            round(soc6_tot * df["soc6_share"]),
            df[c]
        )

    # Create percent-of-workforce columns from the reallocated totals
    # Build pct_map and df_map dynamically for national + all states
    abbrev_to_name = {v: k for k, v in STATE_ABBREV.items()}
    pct_map = {}
    df_map = {}
    for col in emp_cols:
        pct_col = col.replace("emp_tot_", "emp_pct_")
        pct_map[col] = pct_col
        if col == "emp_tot_nat_2024":
            df_map[col] = nat_emp_df_2024
        elif col == "emp_tot_nat_2015":
            df_map[col] = nat_emp_df_2015
        elif col.endswith("_2024"):
            df_map[col] = state_emp_2024_df
        elif col.endswith("_2015"):
            df_map[col] = state_emp_2015_df

    for tot_col, pct_col in pct_map.items():
        if tot_col in df.columns:
            source_df = df_map[tot_col]
            # For national columns, use the single "All Occupations" row
            if "_nat_" in tot_col:
                total_sum = source_df[source_df["OCC_TITLE"] == "All Occupations"]["TOT_EMP"].iloc[0]
            else:
                # Extract state abbrev from column name (e.g. emp_tot_ca_2024 -> ca)
                st_abbrev = tot_col.split("_")[2]
                # 2024 data uses AREA_TITLE (full name), 2015 data uses ST (abbreviation)
                if "AREA_TITLE" in source_df.columns:
                    st_name = abbrev_to_name.get(st_abbrev, "")
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["AREA_TITLE"] == st_name)
                    ]["TOT_EMP"]
                else:
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["ST"] == st_abbrev.upper())
                    ]["TOT_EMP"]
                if state_rows.empty:
                    continue
                total_sum = state_rows.iloc[0]
            df[pct_col] = (df.groupby("title")[tot_col].transform("first") / total_sum) * 100

    df.drop(columns=["soc6","soc6_share"], inplace=True)
    return df


In [600]:
def adjust_emp_new(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2024_df, nat_emp_df_2015, nat_emp_df_2024) -> pd.DataFrame:
    """
    Reallocates employment numbers based on the relative percent of Claude conversations, as we have some duplicate
    6 digit SOC codes but different titles  

    Args:
        tasks_all_wage_emp_df (pd.DataFrame): DataFrame from previous 4.5.

    Returns:
        pd.DataFrame: DataFrame with correct employment numbers
    """

    df = tasks_all_wage_emp_df

    # 6-digit SOC to remove decimals (e.g., '11-1011.03' -> '11-1011')
    df["soc6"] = df["soc_code_2019_full"].astype(str).str[:7]

    # 1. Fill NaNs with the group average (or 0 if the whole group is NaN)
    df["master_pct_normalized"] = df.groupby("soc6")["master_pct_normalized"].transform(lambda x: x.fillna(x.mean())).fillna(0)

    # 2. Dampen extreme values (Square Root) to bring values closer to the mean
    df["master_pct_adj"] = np.sqrt(df["master_pct_normalized"])

    # 3. Calculate shares with a fallback to equal split
    title_val = df.groupby(["soc6", "title_current"])["master_pct_adj"].transform("first")
    soc6_total = df.groupby("soc6")["master_pct_adj"].transform(lambda x: x.unique().sum())
    title_counts = df.groupby("soc6")["title_current"].transform("nunique")

    # If the group total is 0, split equally. Otherwise, use the dampened shares.
    df["soc6_share"] = np.where(
        soc6_total > 0,
        title_val / soc6_total,
        1 / title_counts
    )

    # columns to allocate � national + all states for both years
    emp_cols = [c for c in df.columns if c.startswith("emp_tot_") and c.endswith(("_2024", "_2015"))]

    # For each employment column, check if reallocation is needed
    for c in emp_cols:
        # Get one employment value per title within each soc6
        emp_per_title = df.groupby(["soc6", "title_current"])[c].first().reset_index()
        
        # For each soc6, check if all titles have the same employment (duplicates from fallback)
        def has_duplicate_emp(group):
            return (group[c].nunique() == 1) and (len(group) > 1)
        
        soc6_needs_adjustment = emp_per_title.groupby("soc6").apply(has_duplicate_emp, include_groups=False)
        soc6_to_adjust = soc6_needs_adjustment[soc6_needs_adjustment].index.tolist()
        
        # Create mask for rows that need adjustment
        needs_adjustment = df["soc6"].isin(soc6_to_adjust)
        
        # Get the soc6 total by SUMMING across titles (not max!)
        emp_sum_per_title = df.groupby(["soc6", "title_current"])[c].first()
        soc6_sum = emp_sum_per_title.groupby("soc6").sum()
        soc6_tot_corrected = soc6_sum 
        soc6_tot = df["soc6"].map(soc6_tot_corrected)  # Map back to dataframe
        
        # Only reallocate where needed, otherwise keep original
        df[c] = np.where(
            needs_adjustment,
            round(soc6_tot * df["soc6_share"]),
            df[c]
        )

    # Create percent-of-workforce columns from the reallocated totals
    # Build pct_map and df_map dynamically for national + all states
    abbrev_to_name = {v: k for k, v in STATE_ABBREV.items()}
    pct_map = {}
    df_map = {}
    for col in emp_cols:
        pct_col = col.replace("emp_tot_", "emp_pct_")
        pct_map[col] = pct_col
        if col == "emp_tot_nat_2024":
            df_map[col] = nat_emp_df_2024
        elif col == "emp_tot_nat_2015":
            df_map[col] = nat_emp_df_2015
        elif col.endswith("_2024"):
            df_map[col] = state_emp_2024_df
        elif col.endswith("_2015"):
            df_map[col] = state_emp_2015_df

    for tot_col, pct_col in pct_map.items():
        if tot_col in df.columns:
            source_df = df_map[tot_col]
            # For national columns, use the single "All Occupations" row
            if "_nat_" in tot_col:
                total_sum = source_df[source_df["OCC_TITLE"] == "All Occupations"]["TOT_EMP"].iloc[0]
            else:
                # Extract state abbrev from column name (e.g. emp_tot_ca_2024 -> ca)
                st_abbrev = tot_col.split("_")[2]
                # 2024 data uses AREA_TITLE (full name), 2015 data uses ST (abbreviation)
                if "AREA_TITLE" in source_df.columns:
                    st_name = abbrev_to_name.get(st_abbrev, "")
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["AREA_TITLE"] == st_name)
                    ]["TOT_EMP"]
                else:
                    state_rows = source_df[
                        (source_df["OCC_TITLE"] == "All Occupations") & (source_df["ST"] == st_abbrev.upper())
                    ]["TOT_EMP"]
                if state_rows.empty:
                    continue
                total_sum = state_rows.iloc[0]
            df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100

    df.drop(columns=["soc6","soc6_share"], inplace=True)
    return df


In [601]:

if aei_run or eco_run_2015: 
    tasks_wage_emp_final_df = adjust_emp_old(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2024_df, nat_emp_df_2015, nat_emp_df_2024)
else:
    tasks_wage_emp_final_df = adjust_emp_new(tasks_all_wage_emp_df, state_emp_2015_df, state_emp_2024_df, nat_emp_df_2015, nat_emp_df_2024)

if save_files_each_step:
    tasks_wage_emp_final_df.to_csv("../data/merged_data_files/tasks_wage_emp_final.csv", index=False)


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\286519265.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\286519265.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[pct_col] = (df.groupby("title_current")[tot_col].transform("first") / total_sum) * 100
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\286519265.py:106: PerformanceWarning: DataFrame is highly fragmented.  This is 

## 7: Cleanup On Main Data

In [602]:
def final_cleanup(df) -> pd.DataFrame:
    """
    Description:
        Does some final cleanup and imputing for our data
    
    Args:
        tasks_wage_emp_final_df (pd.DataFrame): DataFrame from Step 5
    
    Returns:
        pd.DataFrame: Final merged DataFrame with all necessary information cleaned.
    """

    # Drop None task row
    # Normalize column names
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["title_normalized"] = df["title"].apply(normalize_text)

    df = df[df["task_normalized"].notna()].copy()
    df["pct_weighted"] = 100 * df["pct_weighted"] / df["pct_weighted"].sum()

    # Get current column order
    cols = df.columns.tolist()

    # Find the index of 'title' and insert 'title_normalized' right after it
    title_idx = cols.index('title')
    cols.insert(title_idx + 1, cols.pop(cols.index('title_normalized')))

    # Reorder the dataframe
    df = df[cols]

    # Fill missing n_responding values with the median within the same occupation
    df["n_responding"] = df.groupby("soc_code_2010")["n_responding"].transform(
        lambda x: x.fillna(x.median()) if not x.isna().all() else x
    )

    # Drop specific confidence interval and bound columns
    cols_to_drop = [
        "task_id", 
        "date",
        "task_type",
        "n_responding",
        "domain_source",
        "n_occurrences",
        "broad_counts",
        "pct_weighted",
        # 2015 Data Columns
        "h_med_nat_nominal_2015",
        "a_med_nat_nominal_2015",
        "h_med_nat_real_2015",
        "a_med_nat_real_2015",
        "h_med_ut_nominal_2015",
        "a_med_ut_nominal_2015",
        "h_med_ut_real_2015",
        "a_med_ut_real_2015",
        "emp_tot_nat_2015",
        "emp_tot_ut_2015",
        "emp_pct_nat_2015",
        "emp_pct_ut_2015",
        # Hourly Wage Columns (2024) � national
        "h_med_nat_2024",
        # emp pct columns � national
        "emp_pct_nat_2024",
    ]
    # Add hourly wage and emp_pct columns for all states (2024)
    for _abbrev in STATE_ABBREV.values():
        cols_to_drop.append(f"h_med_{_abbrev}_2024")
        cols_to_drop.append(f"emp_pct_{_abbrev}_2024")

    df = df.drop(columns=cols_to_drop)

    placeholder_values = ["#", "*", "", "n/a", "na", "--"]
    df.replace(placeholder_values, pd.NA, inplace=True)

    return df


tasks_final_df = final_cleanup(tasks_wage_emp_final_df)

if mcp_run:

    # Load original MCP task results
    mcp_df = pd.read_csv(mcp_data)

    # Normalize merge keys
    mcp_df["task_normalized"] = mcp_df["task"].apply(normalize_text)
    mcp_df["title_current_normalized"] = mcp_df["occupation"].apply(normalize_text)

    tasks_final_df["title_current_normalized"] = (
        tasks_final_df["title_current"].apply(normalize_text)
    )

    # Merge MCP numeric columns back in
    mcp_numeric_cols = [
        "task_normalized",
        "title_current_normalized",
        "n_ratings",
        "mean_rating",
        "median_rating",
        "max_rating",
        "min_rating",
        "p25_rating",
        "p75_rating",
        "top_mcps",
        "top_mcp_urls",
        # "mcp_titles",
    ]

    tasks_final_df = tasks_final_df.merge(
        mcp_df[mcp_numeric_cols],
        on=["task_normalized", "title_current_normalized"],
        how="left"
    )

    tasks_final_df.to_csv(output_file_main_1, index=False)
else:
    tasks_final_df.to_csv(output_file_main_1, index=False)

if save_files_each_step:
    tasks_final_df.to_csv(output_file_main_1, index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3202962429.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["title_normalized"] = df["title"].apply(normalize_text)


In [603]:
# import pandas as pd

# file_path = "../data/TEST_1_tasks_v1.csv"

# # Read only the header
# cols = pd.read_csv(file_path, nrows=0).columns.tolist()

# print(f"Total Columns: {len(cols)}")
# print(cols)

# Main Data Part 2

### Dates, Taxonomy, Physical

In [604]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

file_date_map = {
    "first_pass_microsoft.csv": "2024-09-30",
    "first_pass_aei_v1.csv": "2024-12-23",
    "first_pass_aei_v2.csv": "2025-03-06",
    "first_pass_mcp_v1.csv": "2025-04-24",
    "first_pass_mcp_v2.csv": "2025-05-24",
    "first_pass_mcp_v3.csv": "2025-07-23",
    "first_pass_aei_v3.csv": "2025-08-11",
    "first_pass_aei_api_v3.csv": "2025-08-11",
    "first_pass_aei_v4.csv": "2025-11-13",
    "first_pass_aei_api_v4.csv": "2025-11-13",
    "first_pass_aei_v5.csv": "2026-02-12",
    "first_pass_aei_api_v5.csv": "2026-02-12",
    "first_pass_mcp_v4.csv": "2026-02-18",
    "first_pass_eco_tasks_2015.csv": "2015-01-01",
    "first_pass_eco_tasks_2025.csv": "2025-01-01",
}

dfs = {}

for filename, date in file_date_map.items():
    df = pd.read_csv(data_dir / filename)
    df["date"] = pd.to_datetime(date)
    dfs[filename] = df


data_dir = Path("../data")

# --- LOAD BOTH TAXONOMY FILES ---

# AEI uses v20.1
taxonomy_v20 = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v20.1.csv")
taxonomy_v20["task_normalized"] = taxonomy_v20["task"].apply(normalize_text)
taxonomy_v20["title_normalized"] = taxonomy_v20["Title"].apply(normalize_text)

taxonomy_v20_subset = taxonomy_v20[
    ["task_normalized", "title_normalized", "dwa_title", "iwa_title", "gwa_title"]
].copy()


# MCP + Microsoft use v30.1
taxonomy_v30 = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v30.1.csv")
taxonomy_v30["task_normalized"] = taxonomy_v30["task"].apply(normalize_text)
taxonomy_v30["title_current"] = taxonomy_v30["Title"]

taxonomy_v30_subset = taxonomy_v30[
    ["task_normalized", "title_current", "dwa_title", "iwa_title", "gwa_title"]
].copy()


# --- PROCESS EACH DATASET ---
for filename, df in dfs.items():

    print(f"\nProcessing: {filename}")
    df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    df["task_normalized"] = df["task"].apply(normalize_text)

    if "aei" in filename.lower() or "2015" in filename.lower():
        # AEI merge logic
        df = df.merge(
            taxonomy_v20_subset,
            on=["task_normalized", "title_normalized"],
            how="left"
        )
    elif "microsoft" in filename.lower():
        df = df
    else:
        # MCP  merge logic
        df = df.merge(
            taxonomy_v30_subset,
            on=["task_normalized", "title_current"],
            how="left"
        )

    total = len(df)
    print(filename)
    print(total)

    print(f"DWA missing %: {df['dwa_title'].isna().mean() * 100:.2f}%")
    print(f"IWA missing %: {df['iwa_title'].isna().mean() * 100:.2f}%")
    print(f"GWA missing %: {df['gwa_title'].isna().mean() * 100:.2f}%")

    # Save with suffix
    # new_name = filename.replace(".csv", "_taxonomy.csv")
    # df.to_csv(data_dir / new_name, index=False)

    dfs[filename] = df



data_dir = Path("../data")

phys = pd.read_csv(data_dir / "tasks_dwa_iwa_gwa_v30.1_physical.csv")
phys["task_normalized"] = phys["task"].apply(normalize_text)
phys["title_normalized"] = phys["Title"].apply(normalize_text)
phys["title_current"] = phys["Title"]

def mode_bool(s: pd.Series):
    s = s.dropna()
    if s.empty:
        return pd.NA
    vc = s.value_counts()
    if len(vc) == 1:
        return vc.index[0]
    if vc.iloc[0] == vc.iloc[1]:
        return True
    return vc.idxmax()

phys_pair_aei = (
    phys.groupby(["task_normalized", "title_normalized"], as_index=False)["physical"]
    .agg(mode_bool)
)

phys_pair_non = (
    phys.groupby(["task_normalized", "title_current"], as_index=False)["physical"]
    .agg(mode_bool)
)

task_lookup = phys.groupby("task_normalized")["physical"].agg(mode_bool)

for filename, df in dfs.items():
    print(f"\nProcessing: {filename}")

    # 1) primary merge
    if "aei" in filename.lower() or "2015" in filename.lower():
        df = df.merge(
            phys_pair_aei,
            on=["task_normalized", "title_normalized"],
            how="left",
            validate="m:1",
        )
    else:
        df = df.merge(
            phys_pair_non,
            on=["task_normalized", "title_current"],
            how="left",
            validate="m:1",
        )

    total = len(df)
    print(f"After pair-merge missing %: {df['physical'].isna().mean() * 100:.2f}%")

    df["physical_imputed"] = False
    

    # 2) task-only
    mask = df["physical"].isna()
    if mask.any():
        df.loc[mask, "physical"] = df.loc[mask, "task_normalized"].map(task_lookup)
    print(f"After task-only missing %: {df['physical'].isna().mean() * 100:.2f}%")

    mask_after_task = df["physical"].isna()

    # 3) DWA-majority
    mask = df["physical"].isna()
    if mask.any():
        dwa_majority = (
            df.dropna(subset=["physical", "dwa_title"])
            .groupby("dwa_title")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "dwa_title"].map(dwa_majority)
    print(f"After DWA-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 4) OCCUPATION majority
    mask = df["physical"].isna()
    if mask.any():
        if "aei" in filename.lower() or "2015" in filename.lower():
            occ_col = "title_normalized"
        else:
            occ_col = "title_current"
        occ_majority = (
            df.dropna(subset=["physical", occ_col])
            .groupby(occ_col)["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, occ_col].map(occ_majority)
    print(f"After occupation-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 5) BROAD majority
    mask = df["physical"].isna()
    if mask.any():
        broad_majority = (
            df.dropna(subset=["physical", "broad_occ"])
            .groupby("broad_occ")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "broad_occ"].map(broad_majority)
    print(f"After broad-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 6) MINOR majority
    mask = df["physical"].isna()
    if mask.any() and "minor_occ_category" in df.columns:
        minor_majority = (
            df.dropna(subset=["physical", "minor_occ_category"])
            .groupby("minor_occ_category")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "minor_occ_category"].map(minor_majority)
    print(f"After minor-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    # 7) MAJOR majority
    mask = df["physical"].isna()
    if mask.any() and "major_occ_category" in df.columns:
        major_majority = (
            df.dropna(subset=["physical", "major_occ_category"])
            .groupby("major_occ_category")["physical"]
            .agg(lambda s: bool(s.mean() >= 0.5))
        )
        df.loc[mask, "physical"] = df.loc[mask, "major_occ_category"].map(major_majority)
    print(f"After major-majority missing %: {df['physical'].isna().mean() * 100:.2f}%")

    df.loc[mask_after_task & df["physical"].notna(), "physical_imputed"] = True

    remaining = df["physical"].isna().sum()
    if remaining:
        raise ValueError(f"{filename}: still {remaining} missing physical values after all imputations.")
    else:
        print("All physical values resolved.")
    
    out_name = filename.replace(".csv", "_dates_taxonomy_physical.csv")
    df.to_csv(data_dir / out_name, index=False)

C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["date"] = pd.to_datetime(date)



Processing: first_pass_microsoft.csv
first_pass_microsoft.csv
10438
DWA missing %: 0.00%
IWA missing %: 0.00%
GWA missing %: 0.00%

Processing: first_pass_aei_v1.csv
first_pass_aei_v1.csv
5481
DWA missing %: 3.72%
IWA missing %: 3.72%
GWA missing %: 3.72%

Processing: first_pass_aei_v2.csv
first_pass_aei_v2.csv
5264
DWA missing %: 3.76%
IWA missing %: 3.76%
GWA missing %: 3.76%

Processing: first_pass_mcp_v1.csv
first_pass_mcp_v1.csv
9575
DWA missing %: 0.00%
IWA missing %: 0.00%
GWA missing %: 0.00%

Processing: first_pass_mcp_v2.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

first_pass_mcp_v2.csv
11123
DWA missing %: 0.01%
IWA missing %: 0.01%
GWA missing %: 0.01%

Processing: first_pass_mcp_v3.csv
first_pass_mcp_v3.csv
12500
DWA missing %: 0.01%
IWA missing %: 0.01%
GWA missing %: 0.01%

Processing: first_pass_aei_v3.csv
first_pass_aei_v3.csv
4278
DWA missing %: 3.65%
IWA missing %: 3.65%
GWA missing %: 3.65%

Processing: first_pass_aei_api_v3.csv
first_pass_aei_api_v3.csv
3325
DWA missing %: 3.76%
IWA missing %: 3.76%
GWA missing %: 3.76%

Processing: first_pass_aei_v4.csv
first_pass_aei_v4.csv
5143
DWA missing %: 3.66%
IWA missing %: 3.66%
GWA missing %: 3.66%

Processing: first_pass_aei_api_v4.csv
first_pass_aei_api_v4.csv
3585
DWA missing %: 3.77%
IWA missing %: 3.77%
GWA missing %: 3.77%

Processing: first_pass_aei_v5.csv
first_pass_aei_v5.csv
5217
DWA missing %: 3.64%
IWA missing %: 3.64%
GWA missing %: 3.64%

Processing: first_pass_aei_api_v5.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["task_normalized"] = df["task"].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

...

C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False



Processing: first_pass_eco_tasks_2015.csv
After pair-merge missing %: 28.14%
After task-only missing %: 12.69%
After DWA-majority missing %: 0.93%
After occupation-majority missing %: 0.00%
After broad-majority missing %: 0.00%
After minor-majority missing %: 0.00%
After major-majority missing %: 0.00%
All physical values resolved.


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False



Processing: first_pass_eco_tasks_2025.csv
After pair-merge missing %: 0.00%
After task-only missing %: 0.00%
After DWA-majority missing %: 0.00%
After occupation-majority missing %: 0.00%
After broad-majority missing %: 0.00%
After minor-majority missing %: 0.00%
After major-majority missing %: 0.00%
All physical values resolved.


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\4243987091.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["physical_imputed"] = False


### Microsoft Standardized Auto/Aug Column 

In [605]:
"""
Compute Microsoft automation/augmentation mean score using conditional MCP distributions.
Merges back to task level for the main pipeline.
"""
# ============================================================
# CONFIG
# ============================================================
INCLUDE_EXTRA_STATS = False  # Set True to include max, min, p75, p25, pct_1-5 in output

data_dir = Path("../data")
N_TOTAL = 100_000

# ============================================================
# PART 1: Build conditional distributions from MCP data
# ============================================================

mcp_df = pd.read_csv(data_dir / "mcp_results_2026-02-18.csv")

mcp_dists = []
for idx, cell in mcp_df["task_rating_response_raw"].dropna().items():
    cell = cell.replace("<ratings>", "").replace("</ratings>", "").strip()
    ratings = list(map(int, re.findall(r":(\d+)", cell)))
    if len(ratings) == 0:
        continue
    counts = Counter(ratings)
    n = len(ratings)
    n_mod_plus = counts.get(3, 0) + counts.get(4, 0) + counts.get(5, 0)
    mcp_dists.append({
        'mcp_idx': idx,
        'n_ratings': n,
        'pct_mod_plus': n_mod_plus / n if n > 0 else 0,
        'count_1': counts.get(1, 0),
        'count_2': counts.get(2, 0),
        'count_3': counts.get(3, 0),
        'count_4': counts.get(4, 0),
        'count_5': counts.get(5, 0),
    })

mcp_dist_df = pd.DataFrame(mcp_dists)

bins = np.arange(0, 1.05, 0.05)
bin_labels = [f"{b:.2f}-{b+0.05:.2f}" for b in bins[:-1]]
mcp_dist_df['scope_bin'] = pd.cut(mcp_dist_df['pct_mod_plus'], bins=bins, labels=bin_labels, include_lowest=True)

conditional_dists = {}
for bin_label, group in mcp_dist_df.groupby('scope_bin', observed=True):
    total_counts = group[['count_1', 'count_2', 'count_3', 'count_4', 'count_5']].sum()
    total = total_counts.sum()
    if total > 0:
        conditional_dists[bin_label] = {
            'dist': {
                1: total_counts['count_1'] / total,
                2: total_counts['count_2'] / total,
                3: total_counts['count_3'] / total,
                4: total_counts['count_4'] / total,
                5: total_counts['count_5'] / total,
            },
            'n_mcps': len(group),
            'n_ratings': int(total),
        }

print(f"MCP conditional distributions built: {len(conditional_dists)} bins from {len(mcp_dist_df)} MCPs")

# ============================================================
# PART 2: Apply to Microsoft data
# ============================================================

ms_df = pd.read_csv(data_dir / "first_pass_microsoft_dates_taxonomy_physical.csv")

iwa_agg = ms_df.groupby('iwa_title').agg(
    impact_scope_ai=('impact_scope_ai', 'first'),
    impact_scope_user=('impact_scope_user', 'first'),
    impact_scope_ai_adj=('impact_scope_ai_adj', 'first'),
    impact_scope_user_adj=('impact_scope_user_adj', 'first'),
    pct_normalized_sum=('pct_normalized', 'sum'),
    n_tasks=('task', 'count'),
    gwa_title=('gwa_title', 'first'),
).reset_index()

iwa_agg['impact_scope_avg'] = iwa_agg['impact_scope_ai_adj'] + iwa_agg['impact_scope_user_adj']
iwa_agg['iwa_n'] = N_TOTAL * iwa_agg['pct_normalized_sum']


def find_matching_bin(scope_val, conditional_dists, bin_labels):
    bin_idx = int(scope_val / 0.05)
    bin_idx = min(bin_idx, len(bin_labels) - 1)
    bin_idx = max(bin_idx, 0)
    target_label = bin_labels[bin_idx]
    if target_label in conditional_dists:
        return target_label
    for offset in range(1, len(bin_labels)):
        for direction in [1, -1]:
            check_idx = bin_idx + offset * direction
            if 0 <= check_idx < len(bin_labels):
                check_label = bin_labels[check_idx]
                if check_label in conditional_dists:
                    return check_label
    return None


def compute_stats(scope, n, conditional_dists, bin_labels, include_extra=False):
    bin_label = find_matching_bin(scope, conditional_dists, bin_labels)
    
    if bin_label is None:
        result = {'auto_aug_mean': np.nan}
        if include_extra:
            result.update({
                'auto_aug_max': np.nan, 'auto_aug_min': np.nan,
                'auto_aug_p75': np.nan, 'auto_aug_p25': np.nan,
                'auto_aug_bin': None,
                'auto_aug_pct_1': np.nan, 'auto_aug_pct_2': np.nan,
                'auto_aug_pct_3': np.nan, 'auto_aug_pct_4': np.nan,
                'auto_aug_pct_5': np.nan,
            })
        return result
    
    dist = conditional_dists[bin_label]['dist']
    ratings = sorted(dist.keys())
    
    # Mean
    mean_score = round(sum(r * dist[r] for r in ratings), 3)
    result = {'auto_aug_mean': mean_score}
    
    if include_extra:
        # Max: highest rating with expected count >= 0.5
        max_score = np.nan
        for r in reversed(ratings):
            if n * dist[r] >= 0.5:
                max_score = r
                break
        if pd.isna(max_score):
            for r in reversed(ratings):
                if n * dist[r] >= 0.1:
                    max_score = r
                    break
        
        # Min: lowest rating with expected count >= 0.5
        min_score = np.nan
        for r in ratings:
            if n * dist[r] >= 0.5:
                min_score = r
                break
        if pd.isna(min_score):
            for r in ratings:
                if n * dist[r] >= 0.1:
                    min_score = r
                    break
        
        # Percentiles
        cumsum = 0
        p25 = np.nan
        p75 = np.nan
        for r in ratings:
            cumsum += dist[r]
            if pd.isna(p25) and cumsum >= 0.25:
                p25 = r
            if pd.isna(p75) and cumsum >= 0.75:
                p75 = r
                break
        
        result.update({
            'auto_aug_max': max_score,
            'auto_aug_min': min_score,
            'auto_aug_p75': p75,
            'auto_aug_p25': p25,
            'auto_aug_bin': bin_label,
            'auto_aug_pct_1': round(dist.get(1, 0), 4),
            'auto_aug_pct_2': round(dist.get(2, 0), 4),
            'auto_aug_pct_3': round(dist.get(3, 0), 4),
            'auto_aug_pct_4': round(dist.get(4, 0), 4),
            'auto_aug_pct_5': round(dist.get(5, 0), 4),
        })
    
    return result


# Apply at IWA level
results = iwa_agg.apply(
    lambda row: pd.Series(compute_stats(
        row['impact_scope_avg'], row['iwa_n'],
        conditional_dists, bin_labels, include_extra=INCLUDE_EXTRA_STATS
    )), axis=1
)

iwa_scored = pd.concat([iwa_agg, results], axis=1)

# Diagnostics
print(f"\nauto_aug_mean at IWA level:")
print(iwa_scored['auto_aug_mean'].describe())
if INCLUDE_EXTRA_STATS:
    for col in ['auto_aug_max', 'auto_aug_min', 'auto_aug_p75', 'auto_aug_p25']:
        print(f"\n{col}:")
        print(iwa_scored[col].describe())

# ============================================================
# PART 3: Map back to task level and save
# ============================================================

merge_cols = ['iwa_title', 'iwa_n', 'impact_scope_avg', 'auto_aug_mean']
if INCLUDE_EXTRA_STATS:
    merge_cols += [c for c in iwa_scored.columns if c.startswith('auto_aug_') and c != 'auto_aug_mean']

df_final = ms_df.merge(iwa_scored[merge_cols], on='iwa_title', how='left')

print(f"\nFinal task-level shape: {df_final.shape}")
print(f"Tasks with scores: {df_final['auto_aug_mean'].notna().sum()}")

df_final.to_csv('../data/second_pass_microsoft.csv', index=False)
print("Saved.")

MCP conditional distributions built: 20 bins from 10133 MCPs

auto_aug_mean at IWA level:
count    136.000000
mean       2.574588
std        0.232378
min        2.021000
25%        2.472000
50%        2.565000
75%        2.746000
max        3.222000
Name: auto_aug_mean, dtype: float64

Final task-level shape: (10438, 137)
Tasks with scores: 10438
Saved.


### AEI Standardized Auto/Aug Columns

In [606]:
import pandas as pd

DATA = "../data/"

pairs = [
    (
        "first_pass_aei_v2_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v2.csv",
        "task_name",
    ),
    (
        "first_pass_aei_api_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_api_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v4.csv",
        "task",
    ),
    (
        "first_pass_aei_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v4.csv",
        "task",
    ),
    (   
        "first_pass_aei_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_v5.csv", 
        "task"
    ),
    (   
        "first_pass_aei_api_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_api_v5.csv", 
        "task"
    )
]

for master_file, collab_file, collab_task_col in pairs:

    master_df = pd.read_csv(DATA + master_file)
    collab_df = pd.read_csv(DATA + collab_file)

    # normalize task text in collab file
    collab_df["task_normalized"] = collab_df[collab_task_col].apply(normalize_text)
    # master_df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    # master_df["task_normalized"] = master_df["task"].apply(normalize_text)

    # find numeric columns for averaging duplicates
    numeric_cols = collab_df.select_dtypes(include="number").columns.tolist()

    # collapse duplicate tasks
    collab_df = (
        collab_df
        .groupby("task_normalized", as_index=False)[numeric_cols]
        .mean()
    )

    merged_df = master_df.merge(
        collab_df,
        on="task_normalized",
        how="left",
        # indicator=True
    )

    # merged_df["collab_merge_match"] = merged_df["_merge"] == "both"
    # merged_df = merged_df.drop(columns="_merge")

    # output_name = "test_" + master_file
    # merged_df.to_csv(DATA + output_name, index=False)

    # print(f"Saved: {output_name}")

In [607]:
empty_cutoff = 0.875
number_needed = 2
replacement_cutoff = 0.99

DATA = "../data/"


pairs = [
    (
        "first_pass_aei_v2_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v2.csv",
        "task_name",
    ),
    (
        "first_pass_aei_api_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_v3_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v3.csv",
        "task",
    ),
    (
        "first_pass_aei_api_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_api_v4.csv",
        "task",
    ),
    (
        "first_pass_aei_v4_dates_taxonomy_physical.csv",
        "automation_vs_augmentation_by_task_v4.csv",
        "task",
    ),
    (   
        "first_pass_aei_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_v5.csv", 
        "task"
    ),
    (   
        "first_pass_aei_api_v5_dates_taxonomy_physical.csv", 
        "automation_vs_augmentation_by_task_api_v5.csv", 
        "task"
    )
]

for master_file, collab_file, collab_task_col in pairs:

    master_df = pd.read_csv(DATA + master_file)
    collab_df = pd.read_csv(DATA + collab_file)

    # normalize task text in collab file
    collab_df["task_normalized"] = collab_df[collab_task_col].apply(normalize_text)
    # master_df.drop(columns=["task_normalized"], inplace=True, errors="ignore")  # drop if exists
    # master_df["task_normalized"] = master_df["task"].apply(normalize_text)

    # find numeric columns for averaging duplicates
    numeric_cols = collab_df.select_dtypes(include="number").columns.tolist()

    # collapse duplicate tasks
    collab_df = (
        collab_df
        .groupby("task_normalized", as_index=False)[numeric_cols]
        .mean()
    )

    merged_df = master_df.merge(
        collab_df,
        on="task_normalized",
        how="left",
        #indicator=True
    )

    # ---------------------------------------------
    # COLLAB NORMALIZATION + IMPUTATION
    # ---------------------------------------------

    method_cols = ["directive","feedback_loop","learning","validation","task_iteration"]
    

    # convert percentages to proportions
    merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
    merged_df[method_cols] = merged_df[method_cols].fillna(0)
    # merged_df[method_cols] = merged_df[method_cols] / 100
    

    print("Rows with no collaboration data:", merged_df["collab_missing"].sum())    

    # create empty column
    if "filtered" in merged_df.columns:
        merged_df["empty"] = merged_df["filtered"]
    else:
        merged_df["empty"] = merged_df[["none","not_classified"]].sum(axis=1)

    # ------------------------------------------------
    # normalize rows where empty < empty_cutoff
    # ------------------------------------------------

    merged_df["method_sum"] = merged_df[method_cols].sum(axis=1)

    low_mask = merged_df["empty"] < empty_cutoff

    merged_df.loc[low_mask, method_cols] = merged_df.loc[low_mask, method_cols].div(
        merged_df.loc[low_mask, "method_sum"], axis=0
    )

    merged_df.drop(columns="method_sum", inplace=True)

    # ------------------------------------------------
    # build lookup tables from low-empty tasks
    # ------------------------------------------------

    low_tasks = merged_df[merged_df["empty"] < empty_cutoff]

    dwa_means = (
        low_tasks
        .groupby("dwa_title")[method_cols]
        .mean()
    )

    dwa_means = dwa_means.div(dwa_means.sum(axis=1), axis=0).fillna(0)

    iwa_means = (
        low_tasks
        .groupby("iwa_title")[method_cols]
        .mean()
    )
    gwa_means = (
        low_tasks
        .groupby("gwa_title")[method_cols]
        .mean()
    )

    iwa_means = iwa_means.div(iwa_means.sum(axis=1), axis=0).fillna(0)
    gwa_means = gwa_means.div(gwa_means.sum(axis=1), axis=0).fillna(0)

    dwa_counts = low_tasks.groupby("dwa_title").size()
    iwa_counts = low_tasks.groupby("iwa_title").size()
    gwa_counts = low_tasks.groupby("gwa_title").size()


    dwa_dict = dwa_means.to_dict("index")
    iwa_dict = iwa_means.to_dict("index")
    gwa_dict = gwa_means.to_dict("index")

    # ------------------------------------------------
    # occupation-level lookup tables
    # ------------------------------------------------

    occ_cols = [
        "soc_code_2010",
        "broad_occ",
        "minor_occ_category",
        "major_occ_category"
    ]

    occ_dicts = {}
    occ_counts = {}

    for col in occ_cols:

        means = (
            low_tasks
            .groupby(col)[method_cols]
            .mean()
        )

        means = means.div(means.sum(axis=1), axis=0).fillna(0)

        counts = low_tasks.groupby(col).size()

        occ_dicts[col] = means.to_dict("index")
        occ_counts[col] = counts

    # ------------------------------------------------
    # IMPUTATION FUNCTION
    # ------------------------------------------------

    iwa_flag = []


    def impute_row(row):

        task = row["task"]
        empty = row["empty"]
        source = None

        dwas = task_dwa_map.get(task, [])
        iwas = task_iwa_map.get(task, [])
        gwas = task_gwa_map.get(task, [])

        current_sum = row[method_cols].sum()

        proportions = None

        # -------------------------------
        # DWA fallback
        # -------------------------------
        for dwa in dwas:
            if dwa in dwa_dict and dwa_counts[dwa] >= number_needed:
                proportions = dwa_dict[dwa]
                source = "dwa"
                break

        # -------------------------------
        # IWA fallback
        # -------------------------------
        if proportions is None:
            for iwa in iwas:
                if iwa in iwa_dict and iwa_counts[iwa] >= number_needed:
                    proportions = iwa_dict[iwa]
                    source = "iwa"
                    break

        # -------------------------------
        # GWA fallback
        # -------------------------------
        if proportions is None:
            for gwa in gwas:
                if gwa in gwa_dict and gwa_counts[gwa] >= number_needed:
                    proportions = gwa_dict[gwa]
                    source = "gwa"
                    break
        
        # -------------------------------
        # OCCUPATION FALLBACKS
        # -------------------------------
        if proportions is None:

            for col in occ_cols:

                key = row[col]

                if key in occ_dicts[col] and occ_counts[col][key] >= number_needed:
                    proportions = occ_dicts[col][key]
                    source = col
                    break

        # -------------------------------
        # final failure case
        # -------------------------------
        if proportions is None:
            iwa_flag.append(task)
            return row


        # ------------------------------------------------
        # mid-empty case (.5–.9)
        # ------------------------------------------------
        if empty_cutoff <= empty < replacement_cutoff:

            remaining = 1 - current_sum

            for col in method_cols:
                row[col] += remaining * proportions.get(col, 0)

        # ------------------------------------------------
        # high-empty case (≥ .9)
        # ------------------------------------------------
        elif empty >= replacement_cutoff:

            for col in method_cols:
                row[col] = proportions.get(col, 0)

        row["imputation_source"] = source
        if empty < empty_cutoff:
            row["imputation_source"] = "none_needed"
            return row
        return row

    task_dwa_map = merged_df.groupby("task")["dwa_title"].unique().to_dict()
    task_iwa_map = merged_df.groupby("task")["iwa_title"].unique().to_dict()
    task_gwa_map = merged_df.groupby("task")["gwa_title"].unique().to_dict()
    merged_df["imputation_source"] = None
    merged_df = merged_df.apply(impute_row, axis=1)

    # ------------------------------------------------
    # VALIDATION
    # ------------------------------------------------

    check = merged_df[method_cols].sum(axis=1)

    print("\nRows summing to 1:", (check.round(5)==1).sum(), "/", len(check))

    # ------------------------------------------------
    # cleanup columns
    # ------------------------------------------------

    drop_cols = [c for c in ["none","not_classified","filtered", "collab_missing", "imputation_source"] if c in merged_df.columns]

    merged_df.drop(columns=drop_cols, inplace=True)


    # ------------------------------------------------
    # AEI 1-5 SCALE COLUMNS
    # ------------------------------------------------

    score_map = {
        "learning": 1,
        "validation": 2,
        "task_iteration": 3,
        "feedback_loop": 4,
        "directive": 5,
    }

    method_cols_ordered = ["learning", "validation", "task_iteration", "feedback_loop", "directive"]

    # mean: weighted average
    merged_df["aei_mean"] = sum(
        merged_df[col] * score for col, score in score_map.items()
    )

    # max: highest category with nonzero presence
    def get_max(row):
        for col in reversed(method_cols_ordered):  # high to low
            if row[col] > 0:
                return score_map[col]
        return 1

    # min: lowest category with nonzero presence
    def get_min(row):
        for col in method_cols_ordered:  # low to high
            if row[col] > 0:
                return score_map[col]
        return 1

    # percentile: cumulative from top, find where cumulative first exceeds threshold
    def get_percentile(row, threshold):
        cumulative = 0
        for col in reversed(method_cols_ordered):
            cumulative += row[col]
            if cumulative >= threshold:
                return score_map[col]
        return score_map[method_cols_ordered[0]]

    merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
    merged_df["aei_min"] = merged_df.apply(get_min, axis=1)
    merged_df["aei_p75"] = merged_df.apply(lambda r: get_percentile(r, 0.25), axis=1)
    merged_df["aei_p25"] = merged_df.apply(lambda r: get_percentile(r, 0.75), axis=1)

    # drop source columns
    drop_cols = method_cols_ordered + ["empty", "aei_max", "aei_min", "aei_p75", "aei_p25"] 
    merged_df.drop(columns=[c for c in drop_cols if c in merged_df.columns], inplace=True)
    merged_df.rename(columns={"aei_mean": "auto_aug_mean"}, inplace=True)

    # ------------------------------------------------
    # save output
    # ------------------------------------------------

    output_name = master_file.replace("first", "second").replace("_dates_taxonomy_physical", "")
    
    merged_df.to_csv(DATA + output_name, index=False)
    print(f"Saved: {output_name}")

C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["empty"] = merged_df["filtered"]
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which ha

Rows with no collaboration data: 0


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:273: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["imputation_source"] = None



Rows summing to 1: 5264 / 5264


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:308: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_mean"] = sum(
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:335: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:336: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consid

...

Saved: second_pass_aei_v5.csv
Rows with no collaboration data: 643


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["collab_missing"] = merged_df[method_cols].isna().all(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["empty"] = merged_df[["none","not_classified"]].sum(axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i


Rows summing to 1: 3737 / 3737


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:308: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_mean"] = sum(
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:335: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df["aei_max"] = merged_df.apply(get_max, axis=1)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\2177193359.py:336: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consid

Saved: second_pass_aei_api_v5.csv


### Cleanup

In [608]:
DATA_DIR = Path("../data")  # Adjust this path if your data is elsewhere

# 1. Process "second_pass_aei" files: Drop physical_imputed
aei_seconds = [f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_aei") and f.endswith(".csv")]
for file in aei_seconds:
    df = pd.read_csv(DATA_DIR / file)
    if "physical_imputed" in df.columns:
        df = df.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"])
    df.to_csv(DATA_DIR / file, index=False)
    print(f"Updated: {file}")

# 2. Process "first_pass_aei_v1": Drop physical_imputed, add empty auto_aug_mean
v1_file = "first_pass_aei_v1_dates_taxonomy_physical.csv"
if os.path.exists(DATA_DIR / v1_file):
    df_v1 = pd.read_csv(DATA_DIR / v1_file)
    df_v1 = df_v1.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"], errors='ignore')
    df_v1["auto_aug_mean"] = None
    df_v1.to_csv(DATA_DIR / "second_pass_aei_v1.csv", index=False)
    print("Created: second_pass_aei_v1.csv")

# 2.5. Process Eco 2015 and 2025
eco_files = [
    ("first_pass_eco_tasks_2015_dates_taxonomy_physical.csv", "second_pass_eco_2015.csv"),
    ("first_pass_eco_tasks_2025_dates_taxonomy_physical.csv", "second_pass_eco_2025.csv")
]

for input_file, output_file in eco_files:
    if os.path.exists(DATA_DIR / input_file):
        df_eco = pd.read_csv(DATA_DIR / input_file)
        
        # Drop the physical imputation column as requested
        df_eco = df_eco.drop(columns=["physical_imputed", "master_pct_normalized", "master_pct_adj"], errors='ignore')
        
        # Add the empty auto_aug_mean column to match the AEI structure
        df_eco["auto_aug_mean"] = None
        
        df_eco.to_csv(DATA_DIR / output_file, index=False)
        print(f"Created: {output_file}")
    else:
        print(f"Warning: {input_file} not found.")

# 3. Process "first_pass_mcp" files: Drop/Rename and save as second_pass
mcp_firsts = [f for f in os.listdir(DATA_DIR) if f.startswith("first_pass_mcp") and "_dates_taxonomy_physical" in f]
for file in mcp_firsts:
    df_mcp = pd.read_csv(DATA_DIR / file)
    
    # Drops
    cols_to_drop = ["physical_imputed", "n_ratings", "median_rating", "max_rating", 
                    "min_rating", "p25_rating", "p75_rating", "master_pct_normalized", "master_pct_adj"]
    df_mcp = df_mcp.drop(columns=[c for c in cols_to_drop if c in df_mcp.columns])
    
    # Rename
    if "mean_rating" in df_mcp.columns:
        df_mcp = df_mcp.rename(columns={"mean_rating": "auto_aug_mean"})
    
    # Save as second_pass
    new_name = file.replace("first", "second").replace("_dates_taxonomy_physical", "")
    df_mcp.to_csv(DATA_DIR / new_name, index=False)
    print(f"Created: {new_name}")

# 4. Process "second_pass_microsoft": Drop specific columns
ms_second = "second_pass_microsoft.csv"
if os.path.exists(DATA_DIR / ms_second):
    df_ms = pd.read_csv(DATA_DIR / ms_second)
    df_ms["title_current_normalized"] = df_ms["title_current"].apply(normalize_text)
    df_ms = df_ms.drop(columns=["physical_imputed", "iwa_n", "impact_scope_avg",
                                'impact_scope_user', 'impact_scope_ai', 'impact_scope_user_adj', 'impact_scope_ai_adj', "master_pct_normalized", "master_pct_adj"], errors='ignore')
    df_ms.to_csv(DATA_DIR / ms_second, index=False)
    print(f"Updated: {ms_second}")

# 5. DELETE all first_pass CSVs with "dates_taxonomy_physical"
# --- WARNING: This permanently deletes files ---
for file in os.listdir(DATA_DIR):
    if file.startswith("first_pass") and "_dates_taxonomy_physical.csv" in file:
        os.remove(DATA_DIR / file)
        print(f"Deleted: {file}")

print("\nCleanup Complete.")

Updated: second_pass_aei_api_v3.csv
Updated: second_pass_aei_api_v4.csv
Updated: second_pass_aei_api_v5.csv
Updated: second_pass_aei_v1.csv
Updated: second_pass_aei_v2.csv
Updated: second_pass_aei_v3.csv
Updated: second_pass_aei_v4.csv
Updated: second_pass_aei_v5.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\1247284016.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_v1["auto_aug_mean"] = None


Created: second_pass_aei_v1.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\1247284016.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eco["auto_aug_mean"] = None


Created: second_pass_eco_2015.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\1247284016.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_eco["auto_aug_mean"] = None


Created: second_pass_eco_2025.csv
Created: second_pass_mcp_v1.csv
Created: second_pass_mcp_v2.csv
Created: second_pass_mcp_v3.csv
Created: second_pass_mcp_v4.csv


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\1247284016.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ms["title_current_normalized"] = df_ms["title_current"].apply(normalize_text)


Updated: second_pass_microsoft.csv
Deleted: first_pass_aei_api_v3_dates_taxonomy_physical.csv
Deleted: first_pass_aei_api_v4_dates_taxonomy_physical.csv
Deleted: first_pass_aei_api_v5_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v1_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v2_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v3_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v4_dates_taxonomy_physical.csv
Deleted: first_pass_aei_v5_dates_taxonomy_physical.csv
Deleted: first_pass_eco_tasks_2015_dates_taxonomy_physical.csv
Deleted: first_pass_eco_tasks_2025_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v1_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v2_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v3_dates_taxonomy_physical.csv
Deleted: first_pass_mcp_v4_dates_taxonomy_physical.csv
Deleted: first_pass_microsoft_dates_taxonomy_physical.csv

Cleanup Complete.


In [609]:
DATA_DIR = Path("../data")

# Define the master desired order
# We split this into segments to make the logic clear
COLS_BASE = ['task', 'task_normalized']
COLS_TAXONOMY = ['dwa_title', 'iwa_title', 'gwa_title']
COLS_TITLES_SOC = [
    'title', 'title_normalized', 'soc_code_2010', 
    'title_current', 'title_current_normalized', 
    'soc_code_2019', 'soc_code_2019_full'
]
COLS_CATEGORIES = ['broad_occ','minor_occ_category', 'major_occ_category']
COLS_METRICS = ['date', 'pct_normalized', 'auto_aug_mean', 'physical']
COLS_ECON = ['a_med_nat_2024']
for _abbrev in STATE_ABBREV.values():
    COLS_ECON.append(f'a_med_{_abbrev}_2024')
COLS_ECON.append('emp_tot_nat_2024')
for _abbrev in STATE_ABBREV.values():
    COLS_ECON.append(f'emp_tot_{_abbrev}_2024')

# Combine into one master list
MASTER_ORDER = COLS_BASE + COLS_TAXONOMY + COLS_TITLES_SOC + COLS_CATEGORIES + COLS_METRICS + COLS_ECON

def reorder_columns(file_path):
    df = pd.read_csv(file_path)
    
    # Filter the master order to only include columns that actually exist in this file
    existing_cols = [c for c in MASTER_ORDER if c in df.columns]
    
    # Check if there are any columns in the dataframe NOT in our master list 
    # (to avoid losing data accidentally)
    extra_cols = [c for c in df.columns if c not in existing_cols]
    
    # Final list: standard order + anything left over
    final_col_order = existing_cols + extra_cols
    
    df = df[final_col_order]
    df.to_csv(file_path, index=False)
    print(f"Reordered: {file_path.name}")

# Process all second_pass files
target_files = [f for f in os.listdir(DATA_DIR) if f.startswith("second_pass") and f.endswith(".csv")]

for filename in target_files:
    reorder_columns(DATA_DIR / filename)

print("\nAll files reordered successfully.")

Reordered: second_pass_aei_api_v3.csv
Reordered: second_pass_aei_api_v4.csv
Reordered: second_pass_aei_api_v5.csv
Reordered: second_pass_aei_v1.csv
Reordered: second_pass_aei_v2.csv
Reordered: second_pass_aei_v3.csv
Reordered: second_pass_aei_v4.csv
Reordered: second_pass_aei_v5.csv
Reordered: second_pass_eco_2015.csv
Reordered: second_pass_eco_2025.csv
Reordered: second_pass_eco_2025_with_task_prop.csv
Reordered: second_pass_mcp_v1.csv
Reordered: second_pass_mcp_v2.csv
Reordered: second_pass_mcp_v3.csv
Reordered: second_pass_mcp_v4.csv
Reordered: second_pass_microsoft.csv

All files reordered successfully.


### Check Columns

In [610]:
print(pd.read_csv("../data/second_pass_microsoft.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_mcp_v2.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_aei_v1.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_eco_2015.csv").columns.tolist())
print(pd.read_csv("../data/second_pass_eco_2025.csv").columns.tolist())


['task', 'task_normalized', 'dwa_title', 'iwa_title', 'gwa_title', 'title', 'title_normalized', 'soc_code_2010', 'title_current', 'title_current_normalized', 'soc_code_2019', 'soc_code_2019_full', 'broad_occ', 'minor_occ_category', 'major_occ_category', 'date', 'pct_normalized', 'auto_aug_mean', 'physical', 'a_med_nat_2024', 'a_med_al_2024', 'a_med_ak_2024', 'a_med_az_2024', 'a_med_ar_2024', 'a_med_ca_2024', 'a_med_co_2024', 'a_med_ct_2024', 'a_med_de_2024', 'a_med_dc_2024', 'a_med_fl_2024', 'a_med_ga_2024', 'a_med_hi_2024', 'a_med_id_2024', 'a_med_il_2024', 'a_med_in_2024', 'a_med_ia_2024', 'a_med_ks_2024', 'a_med_ky_2024', 'a_med_la_2024', 'a_med_me_2024', 'a_med_md_2024', 'a_med_ma_2024', 'a_med_mi_2024', 'a_med_mn_2024', 'a_med_ms_2024', 'a_med_mo_2024', 'a_med_mt_2024', 'a_med_ne_2024', 'a_med_nv_2024', 'a_med_nh_2024', 'a_med_nj_2024', 'a_med_nm_2024', 'a_med_ny_2024', 'a_med_nc_2024', 'a_med_nd_2024', 'a_med_oh_2024', 'a_med_ok_2024', 'a_med_or_2024', 'a_med_pa_2024', 'a_med_ri_

# Main Data Part 3

## Ratings

### Parameter Adjustment

In [629]:
frequency_weights = {
    1: 1 / 260,
    2: 4 / 260,
    3: 24 / 260,
    4: 104 / 260,
    5: 1,
    6: 3,
    7: 8
}


#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text


### Create Task Prop

In [630]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

# 1. Load and unique-ify Eco 2025
eco_2025 = pd.read_csv(DATA_DIR / "second_pass_eco_2025.csv")
eco_2025_unique = eco_2025[["title_current", "task_normalized"]].drop_duplicates()

# 2. Load and unique-ify Eco 2015
eco_2015 = pd.read_csv(DATA_DIR / "second_pass_eco_2015.csv")
eco_2015_unique = eco_2015[["title", "task_normalized"]].drop_duplicates()

# 3. Load Crosswalk
crosswalk = pd.read_csv(DATA_DIR / "2010_to_2019_soc_crosswalk.csv")

# 4. Count tasks in 2025 (The Denominator)
counts_2025 = eco_2025_unique.groupby("title_current").size().reset_index(name="count_2025")

# 5. Map 2015 tasks to 2025 titles using Crosswalk (The Numerator)
# First, link 2015 unique tasks to the crosswalk via the 2010 title
cw_mapped = crosswalk.merge(
    eco_2015_unique, 
    left_on="O*NET-SOC 2010 Title", 
    right_on="title", 
    how="inner"
)

# Now group by the 2019 Title (which matches title_current) to get the 2015 task sum
counts_2015_mapped = (
    cw_mapped.groupby("O*NET-SOC 2019 Title")
    .size()
    .reset_index(name="count_2015")
)

# 6. Merge counts and calculate the proportion
final_counts = counts_2025.merge(
    counts_2015_mapped, 
    left_on="title_current", 
    right_on="O*NET-SOC 2019 Title", 
    how="left"
)

# Fill 0 for cases where a 2019 title has no 2010 ancestor in the crosswalk
final_counts["count_2015"] = final_counts["count_2015"].fillna(0)
final_counts["task_prop"] = final_counts["count_2015"] / final_counts["count_2025"]

# 7. Map the task_prop back to the original Eco 2025 dataframe
eco_2025 = eco_2025.merge(
    final_counts[["title_current", "task_prop"]], 
    on="title_current", 
    how="left"
)

# eco_2025 = eco_2025[["title_current", "task_prop"]].drop_duplicates().sort_values("task_prop", ascending=False)
# # Print a few examples
# print(eco_2025[["title_current", "task_prop"]].drop_duplicates().sort_values("task_prop", ascending=False).head(10))
eco_2025.to_csv(DATA_DIR / "second_pass_eco_2025_with_task_prop.csv", index=False)
eco_2025

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,title_current,soc_code_2019,...,emp_tot_vt_2024,emp_tot_va_2024,emp_tot_wa_2024,emp_tot_wv_2024,emp_tot_wi_2024,emp_tot_wy_2024,emp_tot_gu_2024,emp_tot_pr_2024,emp_tot_vi_2024,task_prop
0,Direct or coordinate an organization's financi...,direct or coordinate an organizations financia...,Direct financial operations.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,863.0,7522.0,6741.0,2263.0,7229.0,114.0,521.0,3810.0,195.0,1.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,863.0,7522.0,6741.0,2263.0,7229.0,114.0,521.0,3810.0,195.0,1.0
2,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,863.0,7522.0,6741.0,2263.0,7229.0,114.0,521.0,3810.0,195.0,1.0
3,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,863.0,7522.0,6741.0,2263.0,7229.0,114.0,521.0,3810.0,195.0,1.0
4,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Develop organizational policies or programs.,"Develop organizational policies, systems, or p...",Developing Objectives and Strategies,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,863.0,7522.0,6741.0,2263.0,7229.0,114.0,521.0,3810.0,195.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23845,"Clean interiors of tank cars or tank trucks, u...",clean interiors of tank cars or tank trucks us...,Clean vessels or marine equipment.,"Clean tools, equipment, facilities, or work ar...",Performing General Physical Activities,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0,1.0
23846,Lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,Measure the level or depth of water or other l...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0,1.0
23847,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0,1.0
23848,"Perform general warehouse activities, such as ...",perform general warehouse activities such as o...,Weigh materials to ensure compliance with spec...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,21.0,286.0,249.0,170.0,110.0,20.0,4.0,66.0,2.0,1.0


### Eco Dfs

In [631]:
DATA_DIR = Path("../data")

# ---------------------------------------------------------
# 1. PREP RATINGS FROM v30.1
# ---------------------------------------------------------
tr_df = pd.read_csv(DATA_DIR / "task_ratings_v30.1.csv")
tr_df["task_normalized"] = tr_df["Task"].apply(normalize_text)
tr_df["title_normalized"] = tr_df["Title"].apply(normalize_text)

# Frequency: weighted mean from category distribution
freq_df = tr_df[tr_df["Scale ID"] == "FT"].copy()
freq_df = freq_df[pd.to_numeric(freq_df["Category"], errors="coerce").notnull()]
freq_df["Category"] = freq_df["Category"].astype(int)
freq_df["weighted_val"] = freq_df["Data Value"] * freq_df["Category"].map(frequency_weights) / 100

freq_agg = (
    freq_df.groupby(["title_normalized", "task_normalized"])["weighted_val"]
    .sum()
    .reset_index()
    .rename(columns={"weighted_val": "freq_mean"})
)

# Importance & Relevance (one row per task already)
imp_df = (
    tr_df[tr_df["Scale ID"] == "IM"][["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "importance"})
)
rel_df = (
    tr_df[tr_df["Scale ID"] == "RT"][["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "relevance"})
)

ratings = freq_agg.merge(imp_df, on=["title_normalized", "task_normalized"], how="outer")
ratings = ratings.merge(rel_df, on=["title_normalized", "task_normalized"], how="outer")

# Task-only averages for fallback
ratings_task_avg = (
    ratings.groupby("task_normalized")[["freq_mean", "importance", "relevance"]]
    .mean()
    .reset_index()
    .rename(columns={"freq_mean": "freq_mean_tavg", "importance": "importance_tavg", "relevance": "relevance_tavg"})
)

print(f"Ratings prepared: {len(ratings)} (title, task) pairs")

# ---------------------------------------------------------
# 2. PROCESS EACH ECO FILE
# ---------------------------------------------------------
RATING_COLS = ["freq_mean", "importance", "relevance"]
MIN_FOR_IMPUTE = 5

eco_configs = [
    ("second_pass_eco_2025_with_task_prop.csv", "title_current"),
    ("second_pass_eco_2015.csv", "title"),
]

for eco_file, title_col in eco_configs:
    print(f"\n{'='*60}")
    print(f"Processing: {eco_file} (title col: {title_col})")

    df = pd.read_csv(DATA_DIR / eco_file)
    
    df["task_normalized"] = df["task"].apply(normalize_text)
    df["_merge_title_norm"] = df[title_col].apply(normalize_text)

    total = len(df)
    print(f"Total rows: {total}")

    # --- MERGE: (title, task) ---
    merged = df.merge(
        ratings,
        left_on=["_merge_title_norm", "task_normalized"],
        right_on=["title_normalized", "task_normalized"],
        how="left",
    )

    # Drop the ratings title_normalized to avoid confusion
    merged.drop(columns=["title_normalized"], inplace=True, errors="ignore")

    matched_title_task = merged["freq_mean"].notna().sum()
    print(f"After (title+task) merge: {matched_title_task}/{total} ({matched_title_task/total:.1%})")

    # --- FALLBACK: task-only average for still-missing ---
    merged = merged.merge(ratings_task_avg, on="task_normalized", how="left")
    for col in RATING_COLS:
        mask = merged[col].isna()
        merged.loc[mask, col] = merged.loc[mask, f"{col}_tavg"]
    merged.drop(columns=[f"{c}_tavg" for c in RATING_COLS], inplace=True)

    matched_after_task = merged["freq_mean"].notna().sum()
    print(f"After task-only fallback: {matched_after_task}/{total} ({matched_after_task/total:.1%})")

    # --- IMPUTATION: Occupation-Level Average ---
    merged["rating_imputed"] = "none"
    
    # Use deduplicated (title, task) pairs for occupation-level stats
    # to avoid taxonomy duplicates inflating counts/means
    deduped_for_occ = merged.drop_duplicates(subset=[title_col, "task_normalized"])
    
    for col in RATING_COLS:
        occ_stats = (
            deduped_for_occ[deduped_for_occ[col].notna()]
            .groupby(title_col)[col]
            .agg(["mean", "count"])
            .rename(columns={"mean": f"{col}_occ_lvl", "count": f"{col}_occ_cnt"})
        )
        merged = merged.merge(occ_stats, on=title_col, how="left")
        
        occ_mask = merged[col].isna() & (merged[f"{col}_occ_cnt"] >= MIN_FOR_IMPUTE)
        merged.loc[occ_mask, col] = merged.loc[occ_mask, f"{col}_occ_lvl"]
        merged.loc[occ_mask & (merged["rating_imputed"] == "none"), "rating_imputed"] = "occupation"
        
        merged.drop(columns=[f"{col}_occ_lvl", f"{col}_occ_cnt"], inplace=True)

    matched_after_occ = merged["freq_mean"].notna().sum()
    print(f"After occupation-level average: {matched_after_occ}/{total} ({matched_after_occ/total:.1%})")

    # --- IMPUTATION: DWA -> IWA -> GWA ---
    for level_col, level_label in [("dwa_title", "dwa"), ("iwa_title", "iwa"), ("gwa_title", "gwa")]:
        for col in RATING_COLS:
            still_missing = merged[col].isna()
            if not still_missing.any():
                continue

            # Compute means and counts from rows that HAVE values at this level
            level_stats = (
                merged[merged[col].notna()]
                .groupby(level_col)[col]
                .agg(["mean", "count"])
                .rename(columns={"mean": f"{col}_lvl", "count": f"{col}_cnt"})
            )

            merged = merged.merge(level_stats, on=level_col, how="left", suffixes=("", f"_{level_label}"))

            fill_mask = merged[col].isna() & (merged[f"{col}_cnt"] >= MIN_FOR_IMPUTE)
            merged.loc[fill_mask, col] = merged.loc[fill_mask, f"{col}_lvl"]

            # Update imputation label (only upgrade, don't overwrite a more specific one)
            newly_filled = fill_mask & merged["rating_imputed"].isin(["none", level_label])
            merged.loc[newly_filled, "rating_imputed"] = level_label

            merged.drop(columns=[f"{col}_lvl", f"{col}_cnt"], inplace=True)

    # --- FALLBACK: major_occ_category -> global mean ---
    for col in RATING_COLS:
        mask = merged[col].isna()
        if not mask.any():
            continue

        major_means = merged[merged[col].notna()].groupby("major_occ_category")[col].mean()
        fill_mask = mask & merged["major_occ_category"].map(major_means).notna()
        merged.loc[fill_mask, col] = merged.loc[fill_mask, "major_occ_category"].map(major_means)
        merged.loc[fill_mask & (merged["rating_imputed"].isin(["none", "major"])), "rating_imputed"] = "major"

    for col in RATING_COLS:
        mask = merged[col].isna()
        if mask.any():
            merged.loc[mask, col] = merged[col].mean()
            merged.loc[mask, "rating_imputed"] = "global"

    # Mark rows that got ratings from the initial merge as "none" (no imputation)
    # Rows still missing after everything stay "none" but with NaN values
    final_missing = merged[RATING_COLS].isna().any(axis=1).sum()
    print(f"After DWA/IWA/GWA imputation: {total - final_missing}/{total} ({(total-final_missing)/total:.1%})")
    print(f"Still missing: {final_missing}")

    # Imputation level distribution
    print(f"\nImputation levels:")
    print(merged["rating_imputed"].value_counts().to_string())

    # --- PENALTY: Halve freq_mean for non-occupation imputations in high-impute titles ---
    # 1. Calculate the percentage of tasks per occupation that have ANY imputation (not 'none')
    # Use deduped pairs for imputation percentage calculation
    deduped_impute = merged.drop_duplicates(subset=[title_col, "task_normalized"])
    title_impute_pct = (
        deduped_impute.groupby(title_col)["rating_imputed"]
        .apply(lambda x: (x != "none").mean())
        .to_dict()
    )
    titles_total_impute_pct = merged[title_col].map(title_impute_pct)
    
    # 2. Identify rows that are specifically NOT 'none' and NOT 'occupation'
    non_occ_impute_mask = ~merged["rating_imputed"].isin(["none", "occupation"])
    
    # 3. Apply penalty: If title is > 50% imputed overall, halve freq for the non-occ imputed rows
    penalty_mask = (titles_total_impute_pct > 0.5) & non_occ_impute_mask
    merged.loc[penalty_mask, "freq_mean"] = merged.loc[penalty_mask, "freq_mean"] / 2
    
    # 4. Diagnostics
    n_penalized = penalty_mask.sum()
    print(f"Penalized {n_penalized} tasks in occupations with > 50% total imputation.")

    # --- CLEANUP & SAVE ---
    merged.drop(columns=["_merge_title_norm", "title_normalized_y", "rating_imputed"], inplace=True, errors="ignore")
    merged.rename(columns={"title_normalized_x": "title_normalized"}, inplace=True)

    if eco_file == "second_pass_eco_2025_with_task_prop.csv":
        # 1. Reload the original task_prop source to ensure we have the clean column
        tp_source = pd.read_csv(DATA_DIR / "second_pass_eco_2025_with_task_prop.csv")
        
        # 2. Get only unique title_current and the task_prop column
        tp_lookup = tp_source[["title_current", "task_prop"]].drop_duplicates()
        
        # 3. Force merge it back into the final 'merged' dataframe
        # We drop task_prop from 'merged' first if it exists to avoid suffixes like _x/_y
        if "task_prop" in merged.columns:
            merged.drop(columns=["task_prop"], inplace=True)
            
        merged = merged.merge(tp_lookup, on="title_current", how="left")
        
        # 4. Save
        merged.to_csv(DATA_DIR / "third_pass_eco_2025.csv", index=False)
    else:
        output_name = eco_file.replace("second_pass_", "third_pass_")
        merged.to_csv(DATA_DIR / output_name, index=False)
    print(f"\nSaved: {output_name} ({len(merged)} rows)")

Ratings prepared: 17951 (title, task) pairs

Processing: second_pass_eco_2025_with_task_prop.csv (title col: title_current)


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3871387620.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_merge_title_norm"] = df[title_col].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3871387620.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["rating_imputed"] = "none"


Total rows: 23850
After (title+task) merge: 22784/23850 (95.5%)
After task-only fallback: 22925/23850 (96.1%)
After occupation-level average: 23323/23850 (97.8%)
After DWA/IWA/GWA imputation: 23850/23850 (100.0%)
Still missing: 0

Imputation levels:
rating_imputed
none          22925
dwa             482
occupation      398
iwa              44
gwa               1
Penalized 527 tasks in occupations with > 50% total imputation.

Saved: third_pass_aei_v4.csv (23850 rows)

Processing: second_pass_eco_2015.csv (title col: title)


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3871387620.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_merge_title_norm"] = df[title_col].apply(normalize_text)
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3871387620.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["rating_imputed"] = "none"


Total rows: 24631
After (title+task) merge: 17660/24631 (71.7%)
After task-only fallback: 21098/24631 (85.7%)
After occupation-level average: 23795/24631 (96.6%)
After DWA/IWA/GWA imputation: 24631/24631 (100.0%)
Still missing: 0

Imputation levels:
rating_imputed
none          21098
occupation     2697
dwa             666
iwa             125
major            44
gwa               1
Penalized 829 tasks in occupations with > 50% total imputation.

Saved: third_pass_eco_2015.csv (24631 rows)


### Merge into second passes

In [632]:
# ---------------------------------------------------------
# Load third pass eco files (with ratings already merged)
# ---------------------------------------------------------
eco_2015 = pd.read_csv(DATA_DIR / "third_pass_eco_2015.csv")
eco_2025 = pd.read_csv(DATA_DIR / "third_pass_eco_2025.csv")

# Build lookup tables: (title/title_current, task_normalized) -> ratings
# Deduplicate to avoid blowing up rows on merge
rating_cols = ["freq_mean", "importance", "relevance"]

lookup_2015 = (
    eco_2015[["title_normalized", "task_normalized"] + rating_cols]
    .drop_duplicates(subset=["title_normalized", "task_normalized"])
)

lookup_2025 = (
    eco_2025[["title_current", "task_normalized"] + rating_cols]
    .drop_duplicates(subset=["title_current", "task_normalized"])
)

# ---------------------------------------------------------
# Process all second pass files
# ---------------------------------------------------------
# second_pass_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_") and f.endswith(".csv") and "eco" not in f])
second_pass_files = ["second_pass_aei_api_v3.csv", "second_pass_aei_api_v4.csv", "second_pass_aei_v1.csv", 
                     "second_pass_aei_v2.csv", "second_pass_aei_v3.csv", "second_pass_aei_v4.csv", 
                     "second_pass_mcp_v1.csv", "second_pass_mcp_v2.csv", "second_pass_mcp_v3.csv", 
                     "second_pass_mcp_v4.csv", "second_pass_microsoft.csv", "second_pass_aei_api_v5.csv", "second_pass_aei_v5.csv"]

for filename in second_pass_files:
    df = pd.read_csv(DATA_DIR / filename)
    
    is_aei = "aei" in filename
    
    if is_aei:
        # AEI: merge on title_normalized + task_normalized using 2015 ratings
        df = df.merge(
            lookup_2015,
            on=["title_normalized", "task_normalized"],
            how="left",
        )
    else:
        # MCP, Microsoft: merge on title_current + task_normalized using 2025 ratings
        df = df.merge(
            lookup_2025,
            on=["title_current", "task_normalized"],
            how="left",
        )
    
    # Reorder: insert freq_mean, importance, relevance after auto_aug_mean
    cols = list(df.columns)
    
    # Remove rating cols from wherever they landed
    for rc in rating_cols:
        cols.remove(rc)
    
    # Find insertion point (right before 'physical')
    phys_idx = cols.index("physical")
    for i, rc in enumerate(rating_cols):
        cols.insert(phys_idx + i, rc)
    
    df = df[cols]
    
    # Report
    matched = df["freq_mean"].notna().sum()
    total = len(df)
    print(f"{filename}: {matched}/{total} ({matched/total:.1%}) have ratings")
    
    output_name = filename.replace("second_pass_", "third_pass_")
    df.to_csv(DATA_DIR / output_name, index=False)
    print(f"  -> Saved: {output_name}")

second_pass_aei_api_v3.csv: 3325/3325 (100.0%) have ratings
  -> Saved: third_pass_aei_api_v3.csv
second_pass_aei_api_v4.csv: 3585/3585 (100.0%) have ratings
  -> Saved: third_pass_aei_api_v4.csv
second_pass_aei_v1.csv: 5481/5481 (100.0%) have ratings
  -> Saved: third_pass_aei_v1.csv
second_pass_aei_v2.csv: 5264/5264 (100.0%) have ratings
  -> Saved: third_pass_aei_v2.csv
second_pass_aei_v3.csv: 4278/4278 (100.0%) have ratings
  -> Saved: third_pass_aei_v3.csv
second_pass_aei_v4.csv: 5143/5143 (100.0%) have ratings
  -> Saved: third_pass_aei_v4.csv
second_pass_mcp_v1.csv: 9575/9575 (100.0%) have ratings
  -> Saved: third_pass_mcp_v1.csv
second_pass_mcp_v2.csv: 11122/11123 (100.0%) have ratings
  -> Saved: third_pass_mcp_v2.csv
second_pass_mcp_v3.csv: 12499/12500 (100.0%) have ratings
  -> Saved: third_pass_mcp_v3.csv
second_pass_mcp_v4.csv: 12647/12648 (100.0%) have ratings
  -> Saved: third_pass_mcp_v4.csv
second_pass_microsoft.csv: 10438/10438 (100.0%) have ratings
  -> Saved: third

### Check Columns

In [633]:
DATA_DIR = Path("../data")
pass_segment = "third_pass"  # Change this to "third_pass", "final", etc.

# Define the specific file identifiers you want to check
files_to_check = [
    f"{pass_segment}_microsoft.csv",
    f"{pass_segment}_mcp_v2.csv",
    f"{pass_segment}_aei_v1.csv",
    f"{pass_segment}_eco_2015.csv",
    f"{pass_segment}_eco_2025.csv"
]

for filename in files_to_check:
    file_path = DATA_DIR / filename
    if file_path.exists():
        cols = pd.read_csv(file_path, nrows=0).columns.tolist()
        print(f"--- {filename} ---")
        print(f"{cols}\n")
    else:
        print(f"File not found: {filename}")

--- third_pass_microsoft.csv ---
['task', 'task_normalized', 'dwa_title', 'iwa_title', 'gwa_title', 'title', 'title_normalized', 'soc_code_2010', 'title_current', 'title_current_normalized', 'soc_code_2019', 'soc_code_2019_full', 'broad_occ', 'minor_occ_category', 'major_occ_category', 'date', 'pct_normalized', 'auto_aug_mean', 'freq_mean', 'importance', 'relevance', 'physical', 'a_med_nat_2024', 'a_med_al_2024', 'a_med_ak_2024', 'a_med_az_2024', 'a_med_ar_2024', 'a_med_ca_2024', 'a_med_co_2024', 'a_med_ct_2024', 'a_med_de_2024', 'a_med_dc_2024', 'a_med_fl_2024', 'a_med_ga_2024', 'a_med_hi_2024', 'a_med_id_2024', 'a_med_il_2024', 'a_med_in_2024', 'a_med_ia_2024', 'a_med_ks_2024', 'a_med_ky_2024', 'a_med_la_2024', 'a_med_me_2024', 'a_med_md_2024', 'a_med_ma_2024', 'a_med_mi_2024', 'a_med_mn_2024', 'a_med_ms_2024', 'a_med_mo_2024', 'a_med_mt_2024', 'a_med_ne_2024', 'a_med_nv_2024', 'a_med_nh_2024', 'a_med_nj_2024', 'a_med_nm_2024', 'a_med_ny_2024', 'a_med_nc_2024', 'a_med_nd_2024', 'a_me

### Rename to final

In [634]:
DATA_DIR = Path("../data")
FINAL_DIR = DATA_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Easy to change these for future runs
current_pass_prefix = "third_pass_" 
new_file_prefix = "final_"

# Find all files starting with the current prefix
files_to_process = [f for f in os.listdir(DATA_DIR) 
                    if f.startswith(current_pass_prefix) and f.endswith(".csv")]

if not files_to_process:
    print(f"No files found starting with '{current_pass_prefix}'")
else:
    for filename in sorted(files_to_process):
        df = pd.read_csv(DATA_DIR / filename)
        
        # Swap out the prefix for the new one
        new_name = filename.replace(current_pass_prefix, new_file_prefix)
        
        df.to_csv(FINAL_DIR / new_name, index=False)
        print(f"Transferred: {filename} -> {new_name}")

    print(f"\nSuccessfully moved {len(files_to_process)} files to {FINAL_DIR}")

Transferred: third_pass_aei_api_v3.csv -> final_aei_api_v3.csv
Transferred: third_pass_aei_api_v4.csv -> final_aei_api_v4.csv
Transferred: third_pass_aei_api_v5.csv -> final_aei_api_v5.csv
Transferred: third_pass_aei_v1.csv -> final_aei_v1.csv
Transferred: third_pass_aei_v2.csv -> final_aei_v2.csv
Transferred: third_pass_aei_v3.csv -> final_aei_v3.csv
Transferred: third_pass_aei_v4.csv -> final_aei_v4.csv
Transferred: third_pass_aei_v5.csv -> final_aei_v5.csv
Transferred: third_pass_eco_2015.csv -> final_eco_2015.csv
Transferred: third_pass_eco_2025.csv -> final_eco_2025.csv
Transferred: third_pass_mcp_v1.csv -> final_mcp_v1.csv
Transferred: third_pass_mcp_v2.csv -> final_mcp_v2.csv
Transferred: third_pass_mcp_v3.csv -> final_mcp_v3.csv
Transferred: third_pass_mcp_v4.csv -> final_mcp_v4.csv
Transferred: third_pass_microsoft.csv -> final_microsoft.csv

Successfully moved 15 files to ..\data\final


## Make Cumulative AEI, Add MCPs To Tasks for MCP Data

later add in the other mcp rating metrics for us to make and mcp chart and explorer

In [635]:
"""
build_cumulative_aei.py — Generates cumulative AEI v1–v4 CSVs.

Logic:
  - Cumulative v1 = just v1
  - Cumulative v2 = union of v1 + v2 tasks; max auto_aug; sum pct_normalized (renormalized)
  - Cumulative v3 = union of v1 + v2 + v3 + API v3; same merge logic
  - Cumulative v4 = union of v1 + v2 + v3 + v4 + API v3 + API v4; same merge logic
  - Cumulative AEI-only v3 = union of v1 + v2 + v3 (no API)
  - Cumulative AEI-only v4 = union of v1 + v2 + v3 + v4 (no API)
  - Cumulative API v4 = union of API v3 + API v4

Merge key: (title, task_normalized, dwa_title, iwa_title, gwa_title)
For matched rows:
  - auto_aug_mean: max across versions
  - pct_normalized: sum across versions, then renormalize so total matches single-snapshot scale
  - date: latest date among matched versions
  - all other columns: carry from latest version (they should be identical across versions for same merge key)

Renormalization: after summing pct_normalized, divide each row's pct by the total sum of pct
across all unique (title, task_normalized) pairs, then multiply by the original total sum
from the latest single snapshot. This preserves the scale.

Usage:
  python build_cumulative_aei.py --data-dir ./data --output-dir ./data
"""

import pandas as pd
import numpy as np
from pathlib import Path


MERGE_KEY = ["title", "task_normalized", "dwa_title", "iwa_title", "gwa_title"]

# Which files feed into each cumulative version
# Original combined (AEI + API) cumulative versions
CUMULATIVE_VERSIONS = {
    1: ["final_aei_v1.csv"],
    2: ["final_aei_v1.csv", "final_aei_v2.csv"],
    3: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", "final_aei_api_v3.csv"],
    4: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", "final_aei_v4.csv",
        "final_aei_api_v3.csv", "final_aei_api_v4.csv"],
    5: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", "final_aei_v4.csv", "final_aei_v5.csv",
        "final_aei_api_v3.csv", "final_aei_api_v4.csv", "final_aei_api_v5.csv"],
}

# AEI-only cumulative versions (no API datasets)
CUMULATIVE_AEI_ONLY = {
    3: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv"],
    4: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", "final_aei_v4.csv"],
    5: ["final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", "final_aei_v4.csv", "final_aei_v5.csv"]
}

# API-only cumulative versions
CUMULATIVE_API_ONLY = {
    4: ["final_aei_api_v3.csv", "final_aei_api_v4.csv"],
    5: ["final_aei_api_v3.csv", "final_aei_api_v4.csv", "final_aei_api_v5.csv"]
}


def load_and_tag(path: Path) -> pd.DataFrame:
    """Load a CSV and parse its date column for comparison."""
    df = pd.read_csv(path)
    df["_source_file"] = path.name
    df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
    return df


def build_cumulative(data_dir: Path, files: list[str]) -> pd.DataFrame:
    """Build one cumulative AEI dataset from a list of source files."""
    # Load and concat all source files
    frames = []
    for f in files:
        p = data_dir / f
        if not p.exists():
            print(f"  WARNING: {p} not found, skipping")
            continue
        frames.append(load_and_tag(p))
    
    if not frames:
        raise FileNotFoundError(f"No source files found for: {files}")
    
    combined = pd.concat(frames, ignore_index=True)
    
    # For each merge key group, collapse to one row:
    #   - auto_aug_mean: max
    #   - pct_normalized: sum
    #   - date: latest
    #   - everything else: from the row with the latest date
    
    # Sort by date descending so "first" gives us the latest
    combined = combined.sort_values("_date_parsed", ascending=False)
    
    # Columns that get special treatment
    special_cols = ["auto_aug_mean", "pct_normalized", "date", "_date_parsed", "_source_file"]
    # All other columns: take from latest row
    other_cols = [c for c in combined.columns if c not in MERGE_KEY and c not in special_cols]
    
    # Build aggregation dict
    agg_dict = {}
    for c in other_cols:
        agg_dict[c] = "first"  # latest row (already sorted desc by date)
    agg_dict["auto_aug_mean"] = "max"
    agg_dict["pct_normalized"] = "sum"
    agg_dict["_date_parsed"] = "max"
    
    grouped = combined.groupby(MERGE_KEY, sort=False).agg(agg_dict).reset_index()
    
    # Set date to the latest date in the cumulative set (not per-row)
    latest_date = combined["_date_parsed"].max()
    grouped["date"] = latest_date.strftime("%Y-%m-%d") if pd.notna(latest_date) else combined["date"].iloc[0]
    
    # Renormalize pct_normalized:
    # Dedup on (title, task_normalized) for the denominator — 
    # a task with multiple DWA/IWA/GWA associations shouldn't inflate the total
    dedup_for_pct = grouped.drop_duplicates(subset=["title", "task_normalized"])
    current_total = dedup_for_pct["pct_normalized"].sum()
    
    # Get the target total from the latest single snapshot included
    latest_file = files[-1]  # last file in the list is the latest
    latest_path = data_dir / latest_file
    if latest_path.exists():
        latest_df = pd.read_csv(latest_path)
        latest_dedup = latest_df.drop_duplicates(subset=["title", "task_normalized"])
        target_total = latest_dedup["pct_normalized"].sum()
    else:
        target_total = current_total  # fallback: no rescaling
    
    if current_total > 0:
        grouped["pct_normalized"] = grouped["pct_normalized"] * (target_total / current_total)
    
    # Clean up temp columns
    grouped = grouped.drop(columns=["_date_parsed", "_source_file"], errors="ignore")
    
    return grouped


def print_stats(df: pd.DataFrame) -> None:
    """Print summary stats for a built cumulative dataset."""
    n_rows = len(df)
    n_tasks = df.drop_duplicates(subset=["title", "task_normalized"]).shape[0]
    n_occs = df["title"].nunique()
    print(f"  Rows: {n_rows}")
    print(f"  Unique (title, task) pairs: {n_tasks}")
    print(f"  Unique occupations: {n_occs}")


# ── Configuration ────────────────────────────────────────────────────────────

data_dir = Path("../data/final")       # Adjust to your actual folder
output_dir = Path("../data/final")
output_dir.mkdir(parents=True, exist_ok=True)

# ── Build original combined cumulative (AEI + API) ──────────────────────────

for version in [1, 2, 3, 4, 5]:
    print(f"\nBuilding cumulative AEI v{version} (combined)...")
    print(f"  Sources: {CUMULATIVE_VERSIONS[version]}")
    df = build_cumulative(data_dir, CUMULATIVE_VERSIONS[version])
    print_stats(df)
    out_path = output_dir / f"final_aei_cumulative_v{version}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

# ── Build AEI-only cumulative (no API) ──────────────────────────────────────

for version in [3, 4, 5]:
    print(f"\nBuilding cumulative AEI-only v{version}...")
    print(f"  Sources: {CUMULATIVE_AEI_ONLY[version]}")
    df = build_cumulative(data_dir, CUMULATIVE_AEI_ONLY[version])
    print_stats(df)
    out_path = output_dir / f"final_aei_cumulative_aei_only_v{version}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

# ── Build API-only cumulative ───────────────────────────────────────────────

for version in [4, 5]:
    print(f"\nBuilding cumulative API-only v{version}...")
    print(f"  Sources: {CUMULATIVE_API_ONLY[version]}")
    df = build_cumulative(data_dir, CUMULATIVE_API_ONLY[version])
    print_stats(df)
    out_path = output_dir / f"final_aei_cumulative_api_only_v{version}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

print("\nDone.")


Building cumulative AEI v1 (combined)...
  Sources: ['final_aei_v1.csv']


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_source_file"] = path.name
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:107: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance

  Rows: 5051
  Unique (title, task) pairs: 4056
  Unique occupations: 743
  Saved: ..\data\final\final_aei_cumulative_v1.csv

Building cumulative AEI v2 (combined)...
  Sources: ['final_aei_v1.csv', 'final_aei_v2.csv']


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_source_file"] = path.name
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

  Rows: 5771
  Unique (title, task) pairs: 4639
  Unique occupations: 781
  Saved: ..\data\final\final_aei_cumulative_v2.csv

Building cumulative AEI v3 (combined)...
  Sources: ['final_aei_v1.csv', 'final_aei_v2.csv', 'final_aei_v3.csv', 'final_aei_api_v3.csv']


...

  Rows: 6584
  Unique (title, task) pairs: 5296
  Unique occupations: 811
  Saved: ..\data\final\final_aei_cumulative_aei_only_v5.csv

Building cumulative API-only v4...
  Sources: ['final_aei_api_v3.csv', 'final_aei_api_v4.csv']
  Rows: 3763
  Unique (title, task) pairs: 2947
  Unique occupations: 663


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_source_file"] = path.name
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

  Saved: ..\data\final\final_aei_cumulative_api_only_v4.csv

Building cumulative API-only v5...
  Sources: ['final_aei_api_v3.csv', 'final_aei_api_v4.csv', 'final_aei_api_v5.csv']


C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_source_file"] = path.name
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_date_parsed"] = pd.to_datetime(df["date"], errors="coerce")
C:\Users\teddy\AppData\Local\Temp\ipykernel_7340\3500991834.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

  Rows: 4110
  Unique (title, task) pairs: 3219
  Unique occupations: 683
  Saved: ..\data\final\final_aei_cumulative_api_only_v5.csv

Done.


## DWS Ratings

In [636]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load ────────────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
dws = pd.read_csv(DATA_DIR / "dws_ratings.csv")

# Drop any prior dws columns from previous runs
eco_2025 = eco_2025[[c for c in eco_2025.columns if not c.startswith("dws_star_rating")]]

# ── 2. Normalize titles ─────────────────────────────────────────
def normalize_title(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

dws["title_norm"] = dws["Occupation Title"].apply(normalize_title)

# ── 3. DWS lookup (one rating per normalized title) ─────────────
dws_lookup = (
    dws[["title_norm", "Star Ratings"]]
    .drop_duplicates(subset="title_norm")
    .rename(columns={"Star Ratings": "dws_star_rating"})
)

# ── 4. Dedupe occs, merge direct matches, impute ────────────────
occ = (
    eco_2025[["title_current", "broad_occ", "minor_occ_category", "major_occ_category"]]
    .drop_duplicates(subset="title_current")
    .copy()
)
occ["title_norm"] = occ["title_current"].apply(normalize_title)
occ = occ.merge(dws_lookup, on="title_norm", how="left").drop(columns="title_norm")

direct = occ["dws_star_rating"].notna().sum()
print(f"Direct match: {direct}/{len(occ)}")

for level_col, level_name in [
    ("broad_occ", "broad"),
    ("minor_occ_category", "minor"),
    ("major_occ_category", "major"),
]:
    known = occ[occ["dws_star_rating"].notna()]
    level_means = known.groupby(level_col)["dws_star_rating"].mean().round().astype(int).to_dict()
    missing = occ["dws_star_rating"].isna()
    filled = occ.loc[missing, level_col].map(level_means)
    fill_mask = missing & filled.notna()
    occ.loc[fill_mask, "dws_star_rating"] = filled[fill_mask]
    print(f"After {level_name}: +{fill_mask.sum()}, {occ['dws_star_rating'].isna().sum()} still missing")

occ["dws_star_rating"] = occ["dws_star_rating"].astype("Int64")

# ── 5. Merge deduped ratings back into eco_2025 and save ────────
eco_2025 = eco_2025.merge(occ[["title_current", "dws_star_rating"]], on="title_current", how="left")

out_path = DATA_DIR / "final" / "final_eco_2025.csv"
eco_2025.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(eco_2025)} rows, {eco_2025['dws_star_rating'].notna().sum()} rated)")


Direct match: 607/923
After broad: +197, 119 still missing
After minor: +103, 16 still missing
After major: +16, 0 still missing
Saved: ..\data\final\final_eco_2025.csv  (23850 rows, 23850 rated)


## Job Zones

In [643]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load ────────────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
jz = pd.read_csv(DATA_DIR / "job_zones_v30.1.csv")

# Drop any prior job_zone column from previous runs
eco_2025 = eco_2025[[c for c in eco_2025.columns if c != "job_zone"]]

# ── 2. Normalize titles ─────────────────────────────────────────
def normalize_title(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

jz["title_norm"] = jz["Title"].apply(normalize_title)

# ── 3. Job zone lookup (one value per normalized title) ─────────
jz_lookup = (
    jz[["title_norm", "Job Zone"]]
    .drop_duplicates(subset="title_norm")
    .rename(columns={"Job Zone": "job_zone"})
)

# ── 4. Dedupe occs, merge ───────────────────────────────────────
occ = (
    eco_2025[["title_current"]]
    .drop_duplicates(subset="title_current")
    .copy()
)
occ["title_norm"] = occ["title_current"].apply(normalize_title)
occ = occ.merge(jz_lookup, on="title_norm", how="left").drop(columns="title_norm")

occ["job_zone"] = occ["job_zone"].astype("Int64")

matched = occ["job_zone"].notna().sum()
print(f"Matched: {matched}/{len(occ)}  ({occ['job_zone'].isna().sum()} missing)")

# ── 5. Merge back into eco_2025 and save ────────────────────────
eco_2025 = eco_2025.merge(occ[["title_current", "job_zone"]], on="title_current", how="left")

out_path = DATA_DIR / "final" / "final_eco_2025.csv"
eco_2025.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(eco_2025)} rows, {eco_2025['job_zone'].notna().sum()} rated)")

Matched: 923/923  (0 missing)
Saved: ..\data\final\final_eco_2025.csv  (23850 rows, 23850 rated)


## Column Reorder

In [644]:
FINAL_DIR = Path("../data/final")

# Columns to move right after 'physical'
rating_cols = ["freq_mean", "importance", "relevance"]
# Extra columns only for eco_2025
eco_2025_extra_cols = ["task_prop", "dws_star_rating", "job_zone"]
# Extra columns only for mcp files
mcp_extra_cols = ["top_mcps", "top_mcp_urls"]
# Extra columns only for cumulative files
cumulative_extra_cols = ["auto_aug_mean", "pct_normalized", "date"]

for filename in sorted(os.listdir(FINAL_DIR)):
    if not filename.endswith(".csv"):
        continue

    filepath = FINAL_DIR / filename
    df = pd.read_csv(filepath)
    cols = list(df.columns)

    # Determine which columns to move for this file
    if filename == "final_eco_2025.csv":
        cols_to_move = rating_cols + eco_2025_extra_cols
    elif filename.startswith("final_mcp"):
        cols_to_move = rating_cols + mcp_extra_cols
    elif "cumulative" in filename:
        cols_to_move = cumulative_extra_cols + rating_cols
    else:
        cols_to_move = rating_cols

    # Only move columns that actually exist in this df
    cols_to_move = [c for c in cols_to_move if c in cols]

    if not cols_to_move:
        print(f"Skipped {filename} — no columns to move")
        continue

    if "physical" not in cols:
        print(f"Skipped {filename} — no 'physical' column found")
        continue

    # Remove the columns to move from their current positions
    remaining = [c for c in cols if c not in cols_to_move]

    # Insert right after 'physical'
    phys_idx = remaining.index("physical")
    new_order = remaining[:phys_idx + 1] + cols_to_move + remaining[phys_idx + 1:]

    df = df[new_order]
    df.to_csv(filepath, index=False)
    print(f"Reordered: {filename} — inserted {cols_to_move} after 'physical'")

print("\nDone!")

Reordered: final_aei_api_v3.csv — inserted ['freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_api_v4.csv — inserted ['freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_api_v5.csv — inserted ['freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_cumulative_aei_only_v3.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_cumulative_aei_only_v4.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_cumulative_aei_only_v5.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_cumulative_api_only_v4.csv — inserted ['auto_aug_mean', 'pct_normalized', 'date', 'freq_mean', 'importance', 'relevance'] after 'physical'
Reordered: final_aei_cumulative_api_only_v5.csv — inserte

# Testing

### Diagnostic

In [1501]:
df = pd.read_csv("../data/third_pass_eco_2025.csv")
# unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
unique_title_current_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,title_current,soc_code_2019,...,physical,a_med_nat_2024,a_med_ut_2024,emp_tot_nat_2024,emp_tot_ut_2024,freq_mean,importance,relevance,rating_imputed,task_prop
0,direct or coordinate an organizations financia...,direct or coordinate an organizations financia...,Direct financial operations.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,False,206420.0,163980.0,172469.0,3980.0,0.669603,4.52,74.44,none,1.0
1,confer with board members organization officia...,confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,False,206420.0,163980.0,172469.0,3980.0,0.625965,4.32,81.71,none,1.0
2,prepare budgets for approval including those f...,prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,False,206420.0,163980.0,172469.0,3980.0,0.353887,4.30,93.41,none,1.0
3,direct plan or implement policies objectives o...,direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,False,206420.0,163980.0,172469.0,3980.0,0.971421,4.24,97.79,none,1.0
6,prepare or present reports concerning activiti...,prepare or present reports concerning activiti...,"Prepare financial documents, reports, or budgets.","Prepare financial documents, reports, or budgets.",Documenting/Recording Information,Chief Executives,chief executives,11-1011.00,Chief Executives,11-1011,...,False,206420.0,163980.0,172469.0,3980.0,0.501522,4.17,92.92,none,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23844,unload cars containing liquids by connecting h...,unload cars containing liquids by connecting h...,Connect hoses to equipment or machinery.,Connect components or supply lines to equipmen...,Handling and Moving Objects,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,True,58070.0,47540.0,10920.0,40.0,1.248477,4.08,64.04,none,1.0
23845,clean interiors of tank cars or tank trucks us...,clean interiors of tank cars or tank trucks us...,Clean vessels or marine equipment.,"Clean tools, equipment, facilities, or work ar...",Performing General Physical Activities,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,True,58070.0,47540.0,10920.0,40.0,0.821421,4.02,44.33,none,1.0
23846,lower gauge rods into tanks or read meters to ...,lower gauge rods into tanks or read meters to ...,Measure the level or depth of water or other l...,"Measure physical characteristics of materials,...",Estimating the Quantifiable Characteristics of...,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,True,58070.0,47540.0,10920.0,40.0,1.993477,3.88,65.00,none,1.0
23847,operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",53-7121,...,True,58070.0,47540.0,10920.0,40.0,2.249240,3.87,47.90,none,1.0


In [1502]:
df = pd.read_csv("../data/final/final_aei_v1.csv")
unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
# unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
# unique_title_current_df
unique_title_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,broad_occ,minor_occ_category,...,pct_normalized,auto_aug_mean,freq_mean,importance,relevance,physical,a_med_nat_2024,a_med_ut_2024,emp_tot_nat_2024,emp_tot_ut_2024
0,"Interpret and explain policies, rules, regulat...",interpret and explain policies rules regulatio...,Communicate organizational policies and proced...,"Explain regulations, policies, or procedures.",Interpreting the Meaning of Information for Ot...,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,0.049488,NaN,0.738910,4.090000,96.730000,False,206420.0,163980.0,172469.0,6480.0
1,"Confer with board members, organization offici...",confer with board members organization officia...,Confer with organizational members to accompli...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,0.021864,NaN,0.625965,4.320000,81.710000,False,206420.0,163980.0,172469.0,6480.0
2,Review reports submitted by staff members to r...,review reports submitted by staff members to r...,Analyze data to inform operational decisions o...,Analyze data to improve operations.,Analyzing Data or Information,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,0.002357,NaN,0.964547,4.070000,96.420000,False,206420.0,163980.0,172469.0,6480.0
3,"Serve as liaisons between organizations, share...",serve as liaisons between organizations shareh...,Liaise between departments or other groups to ...,Communicate with others about operational plan...,"Communicating with Supervisors, Peers, or Subo...",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,0.003797,NaN,0.798675,3.920000,68.360000,False,206420.0,163980.0,172469.0,6480.0
4,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,0.005237,NaN,0.971421,4.240000,97.790000,False,206420.0,163980.0,172469.0,6480.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5474,"Become familiar with digging plans, machine ca...",become familiar with digging plans machine cap...,Maintain professional knowledge or certificati...,Maintain current knowledge in area of expertise.,Updating and Using Relevant Knowledge,Excavating and Loading Machine and Dragline Op...,excavating and loading machine and dragline op...,53-7032.00,Surface Mining Machine Operators and Earth Dri...,Extraction Workers,...,0.003928,NaN,2.179578,3.991250,74.033125,False,47120.0,50220.0,29700.0,380.0
5476,Stop gathering arms when cars are full.,stop gathering arms when cars are full,Operate excavation equipment.,Operate construction or excavation equipment.,Controlling Machines and Processes,"Loading Machine Operators, Underground Mining",loading machine operators underground mining,53-7033.00,Underground Mining Machine Operators,Extraction Workers,...,0.003404,NaN,6.776849,4.390000,44.620000,True,68860.0,68860.0,6130.0,68.0
5477,Collect and test samples of cleaning solutions...,collect and test samples of cleaning solutions...,"Test materials, solutions, or samples.",Test characteristics of materials or products.,"Inspecting Equipment, Structures, or Material",Cleaners of Vehicles and Equipment,cleaners of vehicles and equipment,53-7061.00,Laborers and Material Movers,Material Moving Workers,...,0.002488,NaN,1.912242,3.667200,76.062000,True,35270.0,34170.0,373960.0,4450.0
5479,Stack cargo in locations such as transit sheds...,stack cargo in locations such as transit sheds...,"Load shipments, belongings, or materials.","Load products, materials, or equipment for tra...",Performing General Physical Activities,"Labo

In [1503]:
df = pd.read_csv("../data/third_pass_eco_2015.csv")
unique_title_df = df.drop_duplicates(subset=["task_normalized", "title"])
# unique_title_current_df = df.drop_duplicates(subset=["task_normalized", "title_current"])
unique_title_df

,task,task_normalized,dwa_title,iwa_title,gwa_title,title,title_normalized,soc_code_2010,broad_occ,minor_occ_category,...,auto_aug_mean,physical,a_med_nat_2024,a_med_ut_2024,emp_tot_nat_2024,emp_tot_ut_2024,freq_mean,importance,relevance,rating_imputed
0,Appoint department heads or managers and assig...,appoint department heads or managers and assig...,"Direct organizational operations, projects, or...","Direct organizational operations, activities, ...","Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,NaN,False,206420.0,163980.0,172469.0,6480.0,0.348368,4.09,93.14,none
2,"Direct, plan, or implement policies, objective...",direct plan or implement policies objectives o...,Implement organizational process or policy cha...,Implement procedures or processes.,Making Decisions and Solving Problems,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,NaN,False,206420.0,163980.0,172469.0,6480.0,0.971421,4.24,97.79,none
5,"Deliver speeches, write articles, or present i...",deliver speeches write articles or present inf...,Present information to the public.,Provide information or assistance to the public.,Communicating with Persons Outside Organization,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,NaN,False,206420.0,163980.0,172469.0,6480.0,0.362546,3.94,92.70,none
6,"Prepare budgets for approval, including those ...",prepare budgets for approval including those f...,Prepare operational budgets.,Manage budgets or finances.,"Guiding, Directing, and Motivating Subordinates",Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,NaN,False,206420.0,163980.0,172469.0,6480.0,0.353887,4.30,93.41,none
7,Nominate citizens to boards or commissions.,nominate citizens to boards or commissions,NaN,NaN,NaN,Chief Executives,chief executives,11-1011.00,Chief Executives,Top Executives,...,NaN,False,206420.0,163980.0,172469.0,6480.0,0.391724,4.07,27.05,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24625,Check conditions and weights of vessels to ens...,check conditions and weights of vessels to ens...,Inspect cargo areas for cleanliness or condition.,Inspect facilities or equipment.,"Inspecting Equipment, Structures, or Material","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,NaN,True,58070.0,47540.0,10920.0,40.0,3.606558,4.48,86.24,none
24626,"Operate industrial trucks, tractors, loaders a...",operate industrial trucks tractors loaders and...,Operate vehicles or material-moving equipment.,Operate transportation equipment or vehicles.,"Operating Vehicles, Mechanized Devices, or Equ...","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,NaN,True,58070.0,47540.0,10920.0,40.0,2.945191,4.24,72.56,none
24627,Operate conveyors and equipment to transfer gr...,operate conveyors and equipment to transfer gr...,Operate conveyors or other industrial material...,Operate industrial processing or production eq...,Controlling Machines and Processes,"Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,NaN,True,58070.0,47540.0,10920.0,40.0,2.249240,3.87,47.90,none
24628,Copy and attach load specifications to loaded ...,copy and attach load specifications to loaded ...,Mark materials or objects for identification.,Mark materials or objects for identification.,"Identifying Objects, Actions, and Events","Tank Car, Truck, and Ship Loaders",tank car truck and ship loaders,53-7121.00,"Tank Car, Truck, and Ship Loaders",Material Moving Workers,...,NaN,True,58070.0,47540.0,10920.0,40.0,2.841213,4.43,60.24,none


In [1519]:
import pandas as pd

df = pd.read_csv("../data/third_pass_eco_2015.csv")
df = df.drop_duplicates(subset=["task_normalized", "title"])

# Create a boolean flag for whether a row was imputed
df['is_imputed'] = df['rating_imputed'] != 'none'

# Group by title and calculate the percentage of imputed tasks
title_stats = df.groupby('title').agg(
    total_tasks=('is_imputed', 'count'),
    imputed_tasks=('is_imputed', 'sum')
)
title_stats['pct_imputed'] = title_stats['imputed_tasks'] / title_stats['total_tasks']

# Filter for titles where more than half are imputed
highly_imputed_titles_list = title_stats[title_stats['pct_imputed'] > 0.5].index

print(f"Total unique titles: {len(title_stats)}")
print(f"Titles with >50% imputed tasks: {len(highly_imputed_titles_list)}")

if len(highly_imputed_titles_list) > 0:
    # Filter the main dataframe to only include these specific titles
    high_imp_df = df[df['title'].isin(highly_imputed_titles_list)]
    
    print("\nBreakdown of imputation types for titles with >50% imputation:")
    # We exclude 'none' to see only the imputation methods used
    impute_counts = high_imp_df[high_imp_df['rating_imputed'] != 'none']['rating_imputed'].value_counts()
    print(impute_counts)

    print("\nTop 10 most imputed titles:")
    print(title_stats.loc[highly_imputed_titles_list].sort_values('pct_imputed', ascending=False).head(10))

Total unique titles: 974
Titles with >50% imputed tasks: 74

Breakdown of imputation types for titles with >50% imputation:
rating_imputed
occupation    548
dwa           466
iwa           110
major          36
gwa             1
Name: count, dtype: int64

Top 10 most imputed titles:
                                              total_tasks  imputed_tasks  \
title                                                                      
City and Regional Planning Aides                       12             12   
Computer Operators                                     15             15   
Green Marketers                                        16             16   
Fuel Cell Technicians                                  16             16   
Investment Underwriters                                19             19   
Financial Analysts                                     18             18   
Emergency Medical Technicians and Paramedics           15             15   
Energy Brokers                  

## Test Computation

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path("../data/final")
OUT_DIR = DATA_DIR / "test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# TOGGLES
# ---------------------------------------------------------
use_auto_aug = False          # multiply task comp by auto_aug_mean / 5
method = "freq"               # "freq" or "imp"
geo = "nat"                   # "nat" or "ut"
agg_level = "occupation"      # "occupation", "broad", "minor", "major"

emp_col = f"emp_tot_{geo}_2024"
wage_col = f"a_med_{geo}_2024"

prefix = f"{'aug_' if use_auto_aug else ''}{'freq' if method == 'freq' else 'imp'}_{geo}_{agg_level}"

# Map agg_level to column name
AGG_MAP = {
    "occupation": None,  # special case — handled per-file
    "broad": "broad_occ",
    "minor": "minor_occ_category",
    "major": "major_occ_category",
}

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def compute_task_comp(df, method, use_auto_aug):
    """Compute task completion value per row."""
    if method == "freq":
        tc = df["freq_mean"].copy()
    else:
        tc = df["relevance"] * (2 ** df["importance"])
    
    if use_auto_aug:
        aug_col = "auto_aug_mean"
        tc = tc * (df[aug_col] / 5)
    
    return tc


def dedup_and_compute(df, title_col, method, use_auto_aug):
    """Deduplicate to unique (title, task) pairs, compute task_comp."""
    # Take first values for other columns, but we mainly need the numeric ones
    agg_dict = {}
    for c in [emp_col, wage_col, "broad_occ", "minor_occ_category", "major_occ_category",
              "freq_mean", "importance", "relevance", "auto_aug_mean"]:
        if c in df.columns:
            agg_dict[c] = "first"
    if "auto_aug_mean" in df.columns:
        agg_dict["auto_aug_mean"] = "first"
    
    deduped = df.groupby([title_col, "task_normalized"]).agg(agg_dict).reset_index()
    deduped["task_comp"] = compute_task_comp(deduped, method, use_auto_aug)
    return deduped


def aggregate_results(ai_df, eco_df, title_col, agg_level):
    """
    Compute pct_tasks_affected, workers_affected, wages_affected
    at the specified aggregation level.
    """
    if agg_level == "occupation":
        group_col = title_col
    else:
        group_col = AGG_MAP[agg_level]

    # --- Occupation-level first (needed for workers/wages at all levels) ---
    ai_by_occ = ai_df.groupby(title_col).agg(
        ai_task_comp=("task_comp", "sum"),
    ).reset_index()

    eco_by_occ = eco_df.groupby("title_current").agg(
        eco_task_comp=("task_comp", "sum"),
        **{emp_col: (emp_col, "first")},
        **{wage_col: (wage_col, "first")},
        broad_occ=("broad_occ", "first"),
        minor_occ_category=("minor_occ_category", "first"),
        major_occ_category=("major_occ_category", "first"),
    ).reset_index()

    occ_merged = eco_by_occ.merge(
        ai_by_occ,
        left_on="title_current",
        right_on=title_col,
        how="left",
    )
    if title_col != "title_current" and title_col in occ_merged.columns:
        occ_merged.drop(columns=[title_col], inplace=True)

    occ_merged["ai_task_comp"] = occ_merged["ai_task_comp"].fillna(0)
    occ_merged["pct_tasks_affected"] = (occ_merged["ai_task_comp"] / occ_merged["eco_task_comp"]) * 100
    occ_merged["pct_tasks_affected"] = occ_merged["pct_tasks_affected"].clip(upper=100)
    occ_merged["workers_affected"] = (occ_merged["pct_tasks_affected"] / 100) * occ_merged[emp_col]
    occ_merged["wages_affected"] = (occ_merged["pct_tasks_affected"] / 100) * occ_merged[emp_col] * occ_merged[wage_col]

    if agg_level == "occupation":
        return occ_merged[["title_current", "pct_tasks_affected", "workers_affected", "wages_affected",
                           "ai_task_comp", "eco_task_comp", emp_col, wage_col]].sort_values("pct_tasks_affected", ascending=False)

    # --- Higher aggregation levels ---
    # pct_tasks_affected: sum AI comp / sum eco comp at group level
    ai_by_group = ai_df.groupby(group_col)["task_comp"].sum().reset_index(name="ai_task_comp_group")
    eco_by_group = eco_df.groupby(group_col)["task_comp"].sum().reset_index(name="eco_task_comp_group")

    group_merged = eco_by_group.merge(ai_by_group, on=group_col, how="left")
    group_merged["ai_task_comp_group"] = group_merged["ai_task_comp_group"].fillna(0)
    group_merged["pct_tasks_affected"] = (group_merged["ai_task_comp_group"] / group_merged["eco_task_comp_group"]) * 100
    group_merged["pct_tasks_affected"] = group_merged["pct_tasks_affected"].clip(upper=100)

    # workers_affected and wages_affected: sum from occupation level
    occ_by_group = occ_merged.groupby(group_col).agg(
        workers_affected=("workers_affected", "sum"),
        wages_affected=("wages_affected", "sum"),
    ).reset_index()

    group_merged = group_merged.merge(occ_by_group, on=group_col, how="left")

    return group_merged[[group_col, "pct_tasks_affected", "workers_affected", "wages_affected",
                         "ai_task_comp_group", "eco_task_comp_group"]].sort_values("pct_tasks_affected", ascending=False)


# ---------------------------------------------------------
# 1. PREP ECO BASELINE
# ---------------------------------------------------------
eco_2025 = pd.read_csv(DATA_DIR / "final_eco_2025.csv")
eco_deduped = dedup_and_compute(eco_2025, "title_current", method, False)  # no auto_aug on baseline

# Need group columns on eco for higher-level aggregation
# They're already there from dedup_and_compute's agg

# ---------------------------------------------------------
# 2. PREP CROSSWALK (for AEI)
# ---------------------------------------------------------
crosswalk = pd.read_csv("../data/2010_to_2019_soc_crosswalk.csv")

# ---------------------------------------------------------
# 3. PROCESS EACH FILE
# ---------------------------------------------------------
final_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("final_") and f.endswith(".csv") and "eco" not in f])

for filename in final_files:
    df = pd.read_csv(DATA_DIR / filename)
    is_aei = "aei" in filename

    if is_aei:
        # Dedup on (title, task_normalized) with 2010 titles
        ai_deduped = dedup_and_compute(df, "title", method, use_auto_aug)

        # Merge crosswalk
        # Need soc_code_2010 — get it from original df
        soc_lookup = df[["title", "soc_code_2010"]].drop_duplicates(subset=["title"])
        ai_deduped = ai_deduped.merge(soc_lookup, on="title", how="left")

        ai_deduped = ai_deduped.merge(
            crosswalk[["O*NET-SOC 2010 Code", "O*NET-SOC 2019 Title"]],
            left_on="soc_code_2010",
            right_on="O*NET-SOC 2010 Code",
            how="left",
        )

        # Split count: how many 2019 titles does each 2010 code map to
        split_counts = (
            crosswalk.groupby("O*NET-SOC 2010 Code")["O*NET-SOC 2019 Title"]
            .nunique()
            .reset_index(name="split_count")
        )
        ai_deduped = ai_deduped.merge(split_counts, left_on="soc_code_2010", right_on="O*NET-SOC 2010 Code",
                                       how="left", suffixes=("", "_sc"))
        ai_deduped.drop(columns=[c for c in ai_deduped.columns if c.endswith("_sc")], inplace=True)
        ai_deduped["split_count"] = ai_deduped["split_count"].fillna(1)

        ai_deduped["task_comp"] = ai_deduped["task_comp"] / ai_deduped["split_count"]
        ai_deduped[emp_col] = ai_deduped[emp_col] / ai_deduped["split_count"]

        # Collapse to unique (task_normalized, O*NET-SOC 2019 Title)
        agg_cols = {"task_comp": "sum", emp_col: "sum"}
        for c in ["broad_occ", "minor_occ_category", "major_occ_category", wage_col]:
            if c in ai_deduped.columns:
                agg_cols[c] = "first"

        ai_final = (
            ai_deduped.groupby(["O*NET-SOC 2019 Title", "task_normalized"])
            .agg(agg_cols)
            .reset_index()
            .rename(columns={"O*NET-SOC 2019 Title": "title_current"})
        )
        
        # --- Apply task_prop deflation ---
        task_prop_lookup = eco_2025[["title_current", "task_prop"]].drop_duplicates(subset=["title_current"])
        ai_final = ai_final.merge(task_prop_lookup, on="title_current", how="left")
        ai_final["task_prop"] = ai_final["task_prop"].fillna(1.0).clip(lower=1.0)
        ai_final["task_comp"] = ai_final["task_comp"] / ai_final["task_prop"]
        ai_final.drop(columns=["task_prop"], inplace=True)

        # Fill in group columns from eco where missing
        eco_groups = eco_2025[["title_current", "broad_occ", "minor_occ_category", "major_occ_category"]].drop_duplicates(subset=["title_current"])
        for gc in ["broad_occ", "minor_occ_category", "major_occ_category"]:
            if gc in ai_final.columns:
                mask = ai_final[gc].isna()
                if mask.any():
                    fill = ai_final.loc[mask, ["title_current"]].merge(eco_groups[["title_current", gc]], on="title_current", how="left", suffixes=("_old", ""))
                    ai_final.loc[mask, gc] = fill[gc].values

        title_col_for_agg = "title_current"

    else:
        ai_final = dedup_and_compute(df, "title_current", method, use_auto_aug)
        title_col_for_agg = "title_current"

    # Aggregate and output
    result = aggregate_results(ai_final, eco_deduped, title_col_for_agg, agg_level)

    # Diagnostics
    if "pct_tasks_affected" in result.columns:
        over_100 = (result["ai_task_comp"] > result["eco_task_comp"]).sum()
        print(f"{filename} | {agg_level}: {len(result)} rows, {over_100} at 100% cap")

    out_name = f"{prefix}_{filename}"
    result.to_csv(OUT_DIR / out_name, index=False)

final_aei_api_v3.csv | occupation: 923 rows, 0 at 100% cap
final_aei_api_v4.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v1.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v2.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v3.csv | occupation: 923 rows, 0 at 100% cap
final_aei_v4.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v1.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v2.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v3.csv | occupation: 923 rows, 0 at 100% cap
final_mcp_v4.csv | occupation: 923 rows, 0 at 100% cap
final_microsoft.csv | occupation: 923 rows, 0 at 100% cap
